# 10a. Shannon Entropy Reranking — Base / P2-Q Identity Control (Herbal Supplements)

This notebook implements the prior-free Shannon condition on the frozen Notebook 09 query-only winner pool. It is the Shannon counterpart of Base/P2-Q. The personalization weight is fixed to zero, so the final score reduces to the within-pool normalized retrieval score and the upstream candidate order must remain unchanged.

Strict pre-target history and profile-related fields are loaded only for contract checks, diagnostics, and cross-condition schema validation. They do not enter the score. Brand remains visible as candidate-side evidence, but Brand affinity, profile concentration, temporal preference, and all other same-user prior components are disabled.

The notebook evaluates the exact candidate prefixes at depths 100, 300, 500, 700, and 1,000. Its stored execution covers all 1,968 cases and confirms zero rank and metric differences between the Shannon no-prior output and the frozen Stage-1 order at every depth.

The received file contains 24 executed code cells, 24 output-bearing code cells, and no stored errors. Execution counts are sequential from 2 through 25.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. Condition, Inputs, and Fixed Policy

In [3]:
import json
import math
import os
import re
import time
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

print("Libraries loaded.")
print("Random seed:", RANDOM_SEED)


Libraries loaded.
Random seed: 42


In [4]:
# =========================================================
# Config
# =========================================================
NOTEBOOK_NAME = '10a_no_prior_rerank_shannon_entropy_herbal.ipynb'
CATEGORY_ID = 'herbal'
CATEGORY_FOLDER = 'herbal_supplements'
CATEGORY_LABEL = 'Herbal Supplements'
EXPERIMENT_CONDITION = 's2q_no_prior_reranking'
CANDIDATE_POOL_ROLE = 'baseline_query_only'
USE_USER_PRIOR_FEATURES = False
USER_PRIOR_FEATURE_POLICY = "all_strict_pre_target_prior" if USE_USER_PRIOR_FEATURES else "no_user_prior_features"
SHANNON_FRAMEWORK_VERSION = "winner_aware_shannon_v3_reviewed"
STAGE1_CANDIDATE_CONTRACT = "notebook09_winner_long_pool_exact_k"
ITEM_FACET_EVIDENCE_POLICY = "metadata_core_functional_plus_brand_no_review_derived"
ENTROPY_NORMALIZATION_POLICY = "global_catalog_vocabulary_normalized_entropy_unchanged"
SHANNON_SCORING_POLICY = "within_query_minmax_retrieval_plus_entropy_gated_profile_match"
ALGORITHM_CHANGE_SCOPE = "winner_routing_and_leakage_guardrails_only_no_retuning"
CANDIDATE_SOURCE_LABEL = "resolved_from_notebook09"
CANDIDATE_SOURCE_FORMAT = "notebook09_winner_long_pool"
STAGE = 'stage2_baseline_retrieval_no_prior_rerank_shannon'

RERANKING_METHOD = 'shannon_no_prior'
SHANNON_OUTPUT_RERANK_METHOD = 'shannon_no_prior_rerank'
BASELINE_RERANK_METHOD = "stage1_baseline"
EXPECTED_RERANK_METHODS = [BASELINE_RERANK_METHOD, SHANNON_OUTPUT_RERANK_METHOD]

STAGE1_QUERY_METHOD = "C"
QUERY_METHOD = STAGE1_QUERY_METHOD
QUERY_TEXT_COL = "query"
QUERY_VARIANT = "C"

# Values below are resolved from the Notebook 09 winner contract.
DEFAULT_CANDIDATE_POOL_TYPE = "resolved_from_notebook09"
DEFAULT_RETRIEVAL_METHOD = "resolved_from_notebook09"
DEFAULT_RETRIEVAL_METHOD_LABEL = "resolved_from_notebook09"
CANDIDATE_POOL_TYPE = DEFAULT_CANDIDATE_POOL_TYPE
RETRIEVAL_METHOD = DEFAULT_RETRIEVAL_METHOD
RETRIEVAL_METHOD_LABEL = DEFAULT_RETRIEVAL_METHOD_LABEL


def canonical_nonpersonalized_retrieval_method(value):
    return "" if value is None else str(value).strip()


def canonical_personalized_method(value):
    return "" if value is None else str(value).strip()


POOL_K = 1000
POOL_DEPTHS = [100, 300, 500, 700, 1000]
REPORT_POOL_DEPTH = 1000
RUNTIME_REPORT_POOL_DEPTH = REPORT_POOL_DEPTH
EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]

PRIMARY_STAGE2_METRIC = "NDCG@5"
PRIMARY_STAGE2_METRICS = ["NDCG@5"]
SECONDARY_STAGE2_METRICS = ["HitRate@5", "NDCG@1", "HitRate@1", "MRR@5"]
SUPPLEMENTARY_STAGE2_METRICS = ["NDCG@10", "HitRate@10", "MRR@10"]
COMMON_STAGE2_COMPARISON_METRICS = [
    "NDCG@5", "HitRate@5", "NDCG@1", "HitRate@1", "MRR@5",
    "NDCG@10", "HitRate@10", "MRR@10",
]
PREFERENCE_ALIGNMENT_K = 5
PREFERENCE_DIAGNOSTIC_METRICS = ['weighted_facet_overlap_at_5', 'brand_match_at_5', 'concern_match_at_5', 'ingredient_match_at_5']
PREFERENCE_DIAGNOSTIC_FAMILIES = ['brand', 'concern', 'ingredient']
PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS = {'brand': 1.0, 'concern': 1.0, 'ingredient': 1.0}
UPLIFT_DIAGNOSTIC_METRICS = PREFERENCE_DIAGNOSTIC_METRICS

SHANNON_ALPHA = 0.0
SHANNON_HISTORY_SATURATION_N = 8
REGIME_STRENGTH_CAP = {'cold': 0.0, 'weak': 0.2, 'strong': 0.8}
REGIME_ORDER = ['cold', 'weak', 'strong']
DEDUP_HISTORY_BY_CASE_ITEM = False

BRAND_QUERY_ENABLED = False
BRAND_CANDIDATE_VISIBLE = True
BRAND_RERANKING_ENABLED = bool(USE_USER_PRIOR_FEATURES)
USER_BRAND_AFFINITY_ENABLED = USE_USER_PRIOR_FEATURES
HISTORY_SOURCE = "all_prior" if USE_USER_PRIOR_FEATURES else "none"
HISTORICAL_POPULATION_REVIEW_SIGNALS_IN_USER_PROFILE = False
SHARED_ALL_PRIOR_CONTRACT_ROLE = "not_applicable"

NO_PRIOR_ABLATION = True
NO_PRIOR_FEATURE_POLICY = "Base: Notebook 09 query-only winner + deterministic no-prior Shannon control"
NO_PRIOR_SCORE_POLICY = "All user-profile, profile-concentration, and user-history temporal score components are disabled."
NO_PRIOR_MANIFEST_NOTE = NO_PRIOR_SCORE_POLICY
NO_PRIOR_SHANNON_MANIFEST_NOTE = NO_PRIOR_SCORE_POLICY
NO_PRIOR_SCORE_FEATURES = ["rank", "score"]
NO_PRIOR_SHANNON_FINAL_SCORE_USES_PERSONALIZATION = False
NO_PRIOR_SHANNON_FINAL_SCORE_USES_TEMPORAL_RECENCY = False


EXPECTED_QUERY_COUNT = None
EXPECTED_CANDIDATE_ROWS_TOP1000 = None
EXPECTED_REGIME_COUNTS = {}
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
STRICT_SAMPLE_COUNT_QC = True

RUNTIME_BRANCH = 'baseline_winner_shannon_no_prior'
RUNTIME_ROWS = []
NOTEBOOK_TIMER_START = time.perf_counter()

PROJECT_ROOT = Path('/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements')
os.chdir(PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
output_paths = {}

STAGE1_POOL_DIR = OUTPUTS_DIR / "stage1_candidate_pools"
CANDIDATE_POOL_MANIFEST_PATH = STAGE1_POOL_DIR / f"stage1_candidate_pool_export_manifest_{CATEGORY_ID}.json"
CANDIDATE_POOL_SUMMARY_PATH = STAGE1_POOL_DIR / f"winner_candidate_pool_summary_{CATEGORY_ID}.csv"
CANDIDATE_POOL_PATH = None
QUERY_CACHE_PARQUET = None

STAGE_OUTPUT_DIR = OUTPUTS_DIR / 'stage2_nonpersonalized_rerank/shannon_no_prior'
STAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = STAGE_OUTPUT_DIR

QUERY_CACHE_SUMMARY_PATH = PROJECT_ROOT / 'outputs/query_cache/herbal_query_generation_summary_medium_heavy_dspy.json'
QUERY_CACHE_CONFIG_PATH = PROJECT_ROOT / 'outputs/query_cache/herbal_query_generation_config.json'
PRIOR_HISTORY_PARQUET = PROJECT_ROOT / 'data/processed/user_sampling/herbal_user_prior_review_history.parquet'
ITEM_SCHEMA_PARQUET = PROJECT_ROOT / 'data/processed/items/herbal_item_schema_full.parquet'
ITEM_SCHEMA_BASE_PARQUET = PROJECT_ROOT / 'data/processed/items/herbal_item_schema.parquet'
ITEM_DOCS_PARQUET = PROJECT_ROOT / 'data/processed/items/item_docs_herbal.parquet'
ITEMS_FACETS_PARQUET = PROJECT_ROOT / 'data/processed/items/herbal_items_facets.parquet'
PROCESSED_ITEMS_DIR = ITEM_SCHEMA_PARQUET.parent

PERSONALIZED_RETRIEVAL_DIR = OUTPUTS_DIR / "stage1_personalized_retrieval"
SOURCE_PER_QUERY_METRICS_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_per_query_metrics_{CATEGORY_ID}.parquet"
SOURCE_RESULTS_BY_REGIME_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_results_by_regime_{CATEGORY_ID}.csv"
SOURCE_CONFIG_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_config_snapshot_{CATEGORY_ID}.json"
SOURCE_MANIFEST_PATH = PERSONALIZED_RETRIEVAL_DIR / f"personalized_retrieval_manifest_{CATEGORY_ID}.json"


print("Notebook:", NOTEBOOK_NAME)
print("Project root:", PROJECT_ROOT)
print("Experiment condition:", EXPERIMENT_CONDITION)
print("Candidate pool role:", CANDIDATE_POOL_ROLE)
print("User-prior features:", USE_USER_PRIOR_FEATURES)
print("Notebook 09 manifest:", CANDIDATE_POOL_MANIFEST_PATH)
print("Output directory:", OUT_DIR)
print("POOL_DEPTHS:", POOL_DEPTHS)
print("EVAL_KS:", EVAL_KS)


Notebook: 10a_no_prior_rerank_shannon_entropy_herbal.ipynb
Project root: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements
Experiment condition: s2q_no_prior_reranking
Candidate pool role: baseline_query_only
User-prior features: False
Notebook 09 manifest: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/stage1_candidate_pool_export_manifest_herbal.json
Output directory: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior
POOL_DEPTHS: [100, 300, 500, 700, 1000]
EVAL_KS: [1, 5, 10, 100, 300, 500, 700, 1000]


## 2. Runtime Scope

Runtime covers Stage-2 feature preparation, deterministic scoring, and ranking on the already retrieved candidate pool. It excludes Stage-1 retrieval. The measurements are method-specific diagnostics and do not establish a scope-matched serving-cost ranking across rerankers.

In [5]:
def record_runtime_step(
    step_name,
    runtime_sec,
    *,
    method=None,
    branch=None,
    pool_depth=None,
    n_queries=None,
    n_candidates=None,
    runtime_measurement_type='wall_clock_step',
    derived_from_existing_runtime=False,
    error='',
    **extra_metadata,
):
    n_queries_value = np.nan if n_queries is None else n_queries
    n_candidates_value = np.nan if n_candidates is None else n_candidates
    runtime_sec_value = np.nan if runtime_sec is None else float(runtime_sec)

    row = {
        'category': CATEGORY_FOLDER,
        'category_id': CATEGORY_ID,
        'notebook_name': NOTEBOOK_NAME,
        'method': method or RERANKING_METHOD,
        'branch': branch or RUNTIME_BRANCH,
        'step_name': step_name,
        'pool_depth': pool_depth,
        'n_queries': n_queries_value,
        'n_candidates': n_candidates_value,
        'runtime_sec': runtime_sec_value,
        'runtime_sec_per_query': runtime_sec_value / n_queries_value if pd.notna(runtime_sec_value) and pd.notna(n_queries_value) and n_queries_value else np.nan,
        'runtime_sec_per_candidate': runtime_sec_value / n_candidates_value if pd.notna(runtime_sec_value) and pd.notna(n_candidates_value) and n_candidates_value else np.nan,
        'error': error,
        'runtime_measurement_type': runtime_measurement_type,
        'derived_from_existing_runtime': bool(derived_from_existing_runtime),
    }
    row.update(extra_metadata)
    RUNTIME_ROWS.append(row)
    return row


@contextmanager
def runtime_step(
    step_name,
    *,
    method=None,
    branch=None,
    pool_depth=None,
    n_queries=None,
    n_candidates=None,
    runtime_measurement_type='wall_clock_step',
    derived_from_existing_runtime=False,
    **extra_metadata,
):
    start_time = time.perf_counter()
    error_message = ''
    try:
        yield
    except Exception as exc:
        error_message = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        record_runtime_step(
            step_name,
            time.perf_counter() - start_time,
            method=method,
            branch=branch,
            pool_depth=pool_depth,
            n_queries=n_queries,
            n_candidates=n_candidates,
            runtime_measurement_type=runtime_measurement_type,
            derived_from_existing_runtime=derived_from_existing_runtime,
            error=error_message,
            **extra_metadata,
        )


def _runtime_sum(runtime_steps_df, step_names, pool_depth=REPORT_POOL_DEPTH, method=RERANKING_METHOD):
    if runtime_steps_df is None or runtime_steps_df.empty:
        return 0.0
    rows = runtime_steps_df.copy()
    if 'method' in rows.columns:
        rows = rows[rows['method'].astype(str).eq(str(method))]
    if 'pool_depth' in rows.columns:
        rows = rows[pd.to_numeric(rows['pool_depth'], errors='coerce').eq(int(pool_depth))]
    rows = rows[rows['step_name'].astype(str).isin(list(step_names))]
    vals = pd.to_numeric(rows['runtime_sec'], errors='coerce').dropna()
    return float(vals.sum()) if len(vals) else 0.0


def _metric_from_summary(summary_df, method_name, metric_col):
    if summary_df is None or summary_df.empty or metric_col not in summary_df.columns:
        return np.nan
    rows = summary_df.copy()
    if 'rerank_method' in rows.columns:
        rows = rows[rows['rerank_method'].astype(str).eq(method_name)]
    if rows.empty:
        return np.nan
    vals = pd.to_numeric(rows[metric_col], errors='coerce').dropna()
    return float(vals.iloc[0]) if len(vals) else np.nan


def export_runtime_logs(output_dir, report_pool_depth=REPORT_POOL_DEPTH):
    output_dir = Path(output_dir)
    export_start = time.perf_counter()

    n_queries_at_report = np.nan
    n_candidates_at_report = np.nan
    if 'per_query_report_df' in globals() and isinstance(per_query_report_df, pd.DataFrame) and not per_query_report_df.empty:
        report_rows = per_query_report_df[per_query_report_df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]
        n_queries_at_report = float(report_rows['query_id'].nunique()) if 'query_id' in report_rows.columns else float(len(report_rows))
        n_candidates_at_report = n_queries_at_report * int(report_pool_depth) if pd.notna(n_queries_at_report) else np.nan
    if 'reranked_candidates_report_df' in globals() and isinstance(reranked_candidates_report_df, pd.DataFrame) and not reranked_candidates_report_df.empty:
        n_candidates_at_report = float(len(reranked_candidates_report_df[reranked_candidates_report_df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]))

    # Add export and total rows after all material outputs have been written.
    # If the material-output cell already recorded export_outputs, avoid a duplicate step row.
    export_runtime_sec = time.perf_counter() - export_start
    existing_export_row = any(
        str(row.get('step_name', '')) == 'export_outputs'
        and int(row.get('pool_depth') or report_pool_depth) == int(report_pool_depth)
        for row in RUNTIME_ROWS
    )
    if not existing_export_row:
        record_runtime_step(
            'export_outputs',
            export_runtime_sec,
            pool_depth=report_pool_depth,
            n_queries=n_queries_at_report,
            n_candidates=n_candidates_at_report,
        )
    total_runtime_sec = time.perf_counter() - NOTEBOOK_TIMER_START
    record_runtime_step(
        'total_notebook',
        total_runtime_sec,
        pool_depth=report_pool_depth,
        n_queries=n_queries_at_report,
        n_candidates=n_candidates_at_report,
        runtime_measurement_type='wall_clock_total',
    )

    runtime_steps_df = pd.DataFrame(RUNTIME_ROWS)
    if not runtime_steps_df.empty:
        runtime_steps_df['pool_depth'] = pd.to_numeric(runtime_steps_df['pool_depth'], errors='coerce')

    online_steps = ['feature_preparation', 'model_scoring', 'ranking_sorting']
    offline_steps = ['model_fit_or_tuning']
    evaluation_export_steps = ['evaluation', 'export_outputs']

    online_runtime_sec = _runtime_sum(runtime_steps_df, online_steps, report_pool_depth)
    offline_runtime_sec = _runtime_sum(runtime_steps_df, offline_steps, report_pool_depth)
    evaluation_export_runtime_sec = _runtime_sum(runtime_steps_df, evaluation_export_steps, report_pool_depth)

    ndcg_at_5 = _metric_from_summary(summary_overall_df, SHANNON_OUTPUT_RERANK_METHOD, 'NDCG@5') if 'summary_overall_df' in globals() else np.nan
    hitrate_at_5 = _metric_from_summary(summary_overall_df, SHANNON_OUTPUT_RERANK_METHOD, 'HitRate@5') if 'summary_overall_df' in globals() else np.nan
    mrr_at_5 = _metric_from_summary(summary_overall_df, SHANNON_OUTPUT_RERANK_METHOD, 'MRR@5') if 'summary_overall_df' in globals() else np.nan
    baseline_ndcg_at_5 = _metric_from_summary(summary_overall_df, BASELINE_RERANK_METHOD, 'NDCG@5') if 'summary_overall_df' in globals() else np.nan
    baseline_hitrate_at_5 = _metric_from_summary(summary_overall_df, BASELINE_RERANK_METHOD, 'HitRate@5') if 'summary_overall_df' in globals() else np.nan
    delta_ndcg_at_5 = ndcg_at_5 - baseline_ndcg_at_5 if pd.notna(ndcg_at_5) and pd.notna(baseline_ndcg_at_5) else np.nan
    delta_hitrate_at_5 = hitrate_at_5 - baseline_hitrate_at_5 if pd.notna(hitrate_at_5) and pd.notna(baseline_hitrate_at_5) else np.nan

    runtime_method_summary_at1000_df = pd.DataFrame([{
        'category': CATEGORY_FOLDER,
        'category_id': CATEGORY_ID,
        'notebook_name': NOTEBOOK_NAME,
        'branch': RUNTIME_BRANCH,
        'method': RERANKING_METHOD,
        'rerank_method': SHANNON_OUTPUT_RERANK_METHOD,
        'pool_depth': int(report_pool_depth),
        'n_queries': n_queries_at_report,
        'n_candidates': n_candidates_at_report,
        'online_operation_runtime_sec': online_runtime_sec,
        'offline_preparation_runtime_sec': offline_runtime_sec,
        'evaluation_export_runtime_sec': evaluation_export_runtime_sec,
        'total_notebook_runtime_sec': total_runtime_sec,
        'runtime_sec_per_query': online_runtime_sec / n_queries_at_report if pd.notna(n_queries_at_report) and n_queries_at_report else np.nan,
        'runtime_sec_per_candidate': online_runtime_sec / n_candidates_at_report if pd.notna(n_candidates_at_report) and n_candidates_at_report else np.nan,
        'queries_per_second': n_queries_at_report / online_runtime_sec if pd.notna(n_queries_at_report) and online_runtime_sec else np.nan,
        'candidates_per_second': n_candidates_at_report / online_runtime_sec if pd.notna(n_candidates_at_report) and online_runtime_sec else np.nan,
        'runtime_scope': 'stage2_online_reranking_excluding_stage1_retrieval',
        'candidate_pool_type': candidate_pool_type_actual if 'candidate_pool_type_actual' in globals() else DEFAULT_CANDIDATE_POOL_TYPE,
        'retrieval_method': retrieval_method_actual if 'retrieval_method_actual' in globals() else DEFAULT_RETRIEVAL_METHOD,
        'retrieval_method_label': retrieval_method_label_actual if 'retrieval_method_label_actual' in globals() else DEFAULT_RETRIEVAL_METHOD_LABEL,
        'reranking_method': RERANKING_METHOD,
        'sample_scope': 'native',
        'common_sample_filtering_introduced': False,
        'primary_metric': PRIMARY_STAGE2_METRIC,
        'ndcg_at_5': ndcg_at_5,
        'hitrate_at_5': hitrate_at_5,
        'mrr_at_5': mrr_at_5,
        'baseline_ndcg_at_5': baseline_ndcg_at_5,
        'baseline_hitrate_at_5': baseline_hitrate_at_5,
        'delta_ndcg_at_5_vs_baseline': delta_ndcg_at_5,
        'delta_hitrate_at_5_vs_baseline': delta_hitrate_at_5,
        'delta_ndcg5_per_100sec_online': (delta_ndcg_at_5 / online_runtime_sec) * 100 if pd.notna(delta_ndcg_at_5) and online_runtime_sec else np.nan,
        'delta_hitrate5_per_100sec_online': (delta_hitrate_at_5 / online_runtime_sec) * 100 if pd.notna(delta_hitrate_at_5) and online_runtime_sec else np.nan,
    }])

    runtime_notebook_summary_df = runtime_method_summary_at1000_df.copy()
    runtime_notebook_summary_df['total_logged_step_runtime_sec'] = pd.to_numeric(runtime_steps_df.get('runtime_sec', pd.Series(dtype='float64')), errors='coerce').sum()

    component_cols = ['feature_preparation', 'model_fit_or_tuning', 'model_scoring', 'ranking_sorting', 'evaluation', 'export_outputs']
    runtime_method_components_at1000_df = pd.DataFrame([{
        'category_id': CATEGORY_ID,
        'category_folder': CATEGORY_FOLDER,
        'method': RERANKING_METHOD,
        'rerank_method': SHANNON_OUTPUT_RERANK_METHOD,
        'pool_depth': int(report_pool_depth),
        'n_queries': n_queries_at_report,
        'feature_preparation_runtime_sec': _runtime_sum(runtime_steps_df, ['feature_preparation'], report_pool_depth),
        'model_fit_or_tuning_runtime_sec': _runtime_sum(runtime_steps_df, ['model_fit_or_tuning'], report_pool_depth),
        'model_scoring_runtime_sec': _runtime_sum(runtime_steps_df, ['model_scoring'], report_pool_depth),
        'ranking_sorting_runtime_sec': _runtime_sum(runtime_steps_df, ['ranking_sorting'], report_pool_depth),
        'evaluation_runtime_sec': _runtime_sum(runtime_steps_df, ['evaluation'], report_pool_depth),
        'export_outputs_runtime_sec': _runtime_sum(runtime_steps_df, ['export_outputs'], report_pool_depth),
        'online_operation_runtime_sec': online_runtime_sec,
        'runtime_scope': 'stage2_online_reranking_excluding_stage1_retrieval',
        'candidate_pool_type': candidate_pool_type_actual if 'candidate_pool_type_actual' in globals() else DEFAULT_CANDIDATE_POOL_TYPE,
        'retrieval_method': retrieval_method_actual if 'retrieval_method_actual' in globals() else DEFAULT_RETRIEVAL_METHOD,
        'retrieval_method_label': retrieval_method_label_actual if 'retrieval_method_label_actual' in globals() else DEFAULT_RETRIEVAL_METHOD_LABEL,
        'reranking_method': RERANKING_METHOD,
    }])

    if 'runtime_by_pool_depth_df' in globals() and isinstance(runtime_by_pool_depth_df, pd.DataFrame) and not runtime_by_pool_depth_df.empty:
        runtime_pool_depth_diagnostic_df = runtime_by_pool_depth_df.copy()
    else:
        runtime_pool_depth_diagnostic_df = pd.DataFrame([{
            'category_id': CATEGORY_ID,
            'method': RERANKING_METHOD,
            'reranker': SHANNON_OUTPUT_RERANK_METHOD,
            'pool_depth': int(report_pool_depth),
            'n_queries': n_queries_at_report,
            'n_candidates': n_candidates_at_report,
        }])
    runtime_pool_depth_diagnostic_df['notebook_name'] = NOTEBOOK_NAME
    runtime_pool_depth_diagnostic_df['branch'] = RUNTIME_BRANCH
    runtime_pool_depth_diagnostic_df['summary_scope'] = 'all_pool_depths'
    runtime_pool_depth_diagnostic_df['sample_scope'] = 'native'
    runtime_pool_depth_diagnostic_df['candidate_pool_type'] = candidate_pool_type_actual if 'candidate_pool_type_actual' in globals() else DEFAULT_CANDIDATE_POOL_TYPE
    runtime_pool_depth_diagnostic_df['retrieval_method'] = retrieval_method_actual if 'retrieval_method_actual' in globals() else DEFAULT_RETRIEVAL_METHOD
    runtime_pool_depth_diagnostic_df['retrieval_method_label'] = retrieval_method_label_actual if 'retrieval_method_label_actual' in globals() else DEFAULT_RETRIEVAL_METHOD_LABEL
    runtime_pool_depth_diagnostic_df['reranking_method'] = RERANKING_METHOD

    runtime_steps_df.to_csv(output_dir / 'runtime_steps.csv', index=False)
    runtime_notebook_summary_df.to_csv(output_dir / 'runtime_notebook_summary.csv', index=False)
    runtime_method_summary_at1000_df.to_csv(output_dir / 'runtime_method_summary_at1000.csv', index=False)
    runtime_pool_depth_diagnostic_df.to_csv(output_dir / 'runtime_pool_depth_diagnostic.csv', index=False)
    runtime_method_components_at1000_df.to_csv(output_dir / f'runtime_method_components_at1000_{CATEGORY_ID}.csv', index=False)

    return (
        runtime_steps_df,
        runtime_notebook_summary_df,
        runtime_method_summary_at1000_df,
        runtime_pool_depth_diagnostic_df,
        runtime_method_components_at1000_df,
    )

print('Runtime framework ready.')


Runtime framework ready.


## 3. Shared Contract Helpers

In [6]:
# Exact-contract helpers

def load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8-sig") as handle:
        return json.load(handle)


def load_json_required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required JSON artifact: {path}")
    with open(path, "r", encoding="utf-8-sig") as handle:
        return json.load(handle)


def make_jsonable(obj):
    if isinstance(obj, dict):
        return {str(key): make_jsonable(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [make_jsonable(value) for value in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.bool_):
        return bool(obj)
    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass
    return obj


def require_columns(frame, columns, label):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def normalize_space(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def normalize_text(value):
    text = normalize_space(value).lower()
    text = re.sub(r"[^a-z0-9+\s_-]", " ", text)
    return re.sub(r"\s+", " ", text.replace("_", " ")).strip()


def normalize_item_id(value):
    return normalize_space(value)


def as_bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0).ne(0)
    return series.fillna("").astype(str).str.strip().str.lower().isin({"1", "true", "t", "yes", "y"})


def to_timestamp_ms(series):
    values = pd.to_numeric(series, errors="coerce")
    positive = values[values.gt(0)]
    if not positive.empty and float(positive.median()) < 1e11:
        values = values * 1000.0
    return values


def safe_numeric(series, default=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(default)


def safe_int_series(series, default=0):
    return pd.to_numeric(series, errors="coerce").fillna(default).astype(int)


def normalize_regime(value):
    value = normalize_text(value)
    return value if value in set(REGIME_ORDER) else ""


def regime_from_prior_count(prior_n):
    prior_n = int(float(prior_n)) if pd.notna(prior_n) else 0
    if prior_n <= 0:
        return "cold"
    if prior_n <= 9:
        return "weak"
    return "strong"


def apply_regime_order(frame, regime_col="regime"):
    if regime_col not in frame.columns:
        return frame
    out = frame.copy()
    values = out[regime_col].map(normalize_regime)
    out[regime_col] = pd.Categorical(values, categories=REGIME_ORDER, ordered=True)
    return out


def minmax_by_group(frame, group_col, value_col, rank_col="rank", max_rank=POOL_K):
    values = pd.to_numeric(frame[value_col], errors="coerce").fillna(0.0)
    group_min = values.groupby(frame[group_col]).transform("min")
    group_max = values.groupby(frame[group_col]).transform("max")
    denominator = (group_max - group_min).replace(0, np.nan)
    normalized = (values - group_min) / denominator
    rank_values = pd.to_numeric(frame[rank_col], errors="coerce").fillna(max_rank).clip(lower=1)
    rank_fallback = 1.0 - ((rank_values - 1.0) / max(max_rank - 1, 1))
    return normalized.where(denominator.notna(), rank_fallback).fillna(0.0).clip(0.0, 1.0)


def metrics_at_rank(rank, ks):
    rank_value = int(rank) if pd.notna(rank) else None
    row = {}
    for k in ks:
        hit = int(rank_value is not None and rank_value <= int(k))
        row[f"HitRate@{k}"] = float(hit)
        row[f"NDCG@{k}"] = float(1.0 / math.log2(rank_value + 1)) if hit else 0.0
        row[f"MRR@{k}"] = float(1.0 / rank_value) if hit else 0.0
    return row


def jaccard(left, right):
    left = set(left or [])
    right = set(right or [])
    return float(len(left & right) / len(left | right)) if left and right else 0.0


print("Exact-contract helpers ready.")


Exact-contract helpers ready.


In [7]:
# Notebook 09 exact candidate schema is enforced in the next input cell.
REQUIRED_NOTEBOOK09_COLUMNS = [
    "category_id", "case_id", "query_id", "user_id", "regime", "sampling_bracket",
    "target_selection_mode", "target_parent_asin", "gt_item_id", "target_timestamp_ms",
    "prior_history_n", "query_text", "candidate_pool_role", "method_slug",
    "retrieval_method", "candidate_parent_asin", "candidate_rank", "candidate_score",
    "candidate_score_source", "is_gt",
]

print("Notebook 09 exact candidate schema registered.")


Notebook 09 exact candidate schema registered.


## 4. Load the Frozen S1-Q Candidate Pool

In [8]:
# =========================================================
# Validate Static Inputs
# =========================================================
required_input_paths = {
    "stage1_candidate_pool_manifest": CANDIDATE_POOL_MANIFEST_PATH,
    "items_facets_parquet": ITEMS_FACETS_PARQUET,
}
if USE_USER_PRIOR_FEATURES:
    required_input_paths["prior_history_parquet"] = PRIOR_HISTORY_PARQUET

optional_input_paths = {
    "candidate_pool_summary": CANDIDATE_POOL_SUMMARY_PATH,
    "item_schema_parquet": ITEM_SCHEMA_PARQUET,
    "item_schema_base_parquet": ITEM_SCHEMA_BASE_PARQUET,
    "item_docs_parquet": ITEM_DOCS_PARQUET,
    "query_cache_summary_path": QUERY_CACHE_SUMMARY_PATH,
    "query_cache_config_path": QUERY_CACHE_CONFIG_PATH,
}
missing_inputs = [name for name, path in required_input_paths.items() if not Path(path).exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

print("Static input paths validated.")
for name, path in {**required_input_paths, **optional_input_paths}.items():
    print(f"{name}: {path} | exists={Path(path).exists()}")


Static input paths validated.
stage1_candidate_pool_manifest: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/stage1_candidate_pool_export_manifest_herbal.json | exists=True
items_facets_parquet: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_items_facets.parquet | exists=True
candidate_pool_summary: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/winner_candidate_pool_summary_herbal.csv | exists=True
item_schema_parquet: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema_full.parquet | exists=True
item_schema_base_parquet: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema.parquet | exists=True
item_docs_parquet: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/item_docs_herbal.parquet | exi

In [9]:
# =========================================================
# Load the exact Notebook 09 winner candidate pool
# =========================================================
load_start = time.perf_counter()

stage1_pool_manifest = load_json_required(CANDIDATE_POOL_MANIFEST_PATH)
if stage1_pool_manifest.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 09 candidate manifest category mismatch.")
if stage1_pool_manifest.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 09 must use the exact-K candidate budget policy.")
if stage1_pool_manifest.get("variable_candidate_count_allowed") is not False:
    raise RuntimeError("Variable candidate counts are not allowed for Stage 2.")

output_paths_09 = stage1_pool_manifest.get("output_paths", {})
input_paths_09 = stage1_pool_manifest.get("input_paths", {})
if CANDIDATE_POOL_ROLE == "baseline_query_only":
    candidate_path_key = "query_only_winner_long"
    CANDIDATE_RETRIEVAL_METHOD_KEY = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_key"))
    CANDIDATE_RETRIEVAL_METHOD_LABEL = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_label"))
elif CANDIDATE_POOL_ROLE == "personalized_retrieval":
    candidate_path_key = "personalized_winner_long"
    CANDIDATE_RETRIEVAL_METHOD_KEY = normalize_space(stage1_pool_manifest.get("selected_personalized_method_slug"))
    CANDIDATE_RETRIEVAL_METHOD_LABEL = normalize_space(stage1_pool_manifest.get("selected_personalized_method_label"))
else:
    raise RuntimeError(f"Unsupported CANDIDATE_POOL_ROLE: {CANDIDATE_POOL_ROLE}")
if not CANDIDATE_RETRIEVAL_METHOD_KEY or not CANDIDATE_RETRIEVAL_METHOD_LABEL:
    raise RuntimeError("Notebook 09 manifest does not contain the selected winner identity.")
SOURCE_BASELINE_METHOD_KEY = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_key"))
SOURCE_BASELINE_METHOD_LABEL = normalize_space(stage1_pool_manifest.get("baseline_retrieval_winner_method_label"))
if not SOURCE_BASELINE_METHOD_KEY or not SOURCE_BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 09 manifest does not contain the query-only baseline winner identity.")
DEFAULT_CANDIDATE_POOL_TYPE = CANDIDATE_RETRIEVAL_METHOD_LABEL
DEFAULT_RETRIEVAL_METHOD = CANDIDATE_RETRIEVAL_METHOD_LABEL
DEFAULT_RETRIEVAL_METHOD_LABEL = CANDIDATE_RETRIEVAL_METHOD_LABEL
CANDIDATE_POOL_TYPE = CANDIDATE_RETRIEVAL_METHOD_LABEL
RETRIEVAL_METHOD = CANDIDATE_RETRIEVAL_METHOD_LABEL
RETRIEVAL_METHOD_LABEL = CANDIDATE_RETRIEVAL_METHOD_LABEL
CANDIDATE_SOURCE_LABEL = f"notebook09_{CANDIDATE_POOL_ROLE}_{CANDIDATE_RETRIEVAL_METHOD_KEY}"

candidate_path_text = normalize_space(output_paths_09.get(candidate_path_key))
query_cache_path_text = normalize_space(input_paths_09.get("query_cache"))
if not candidate_path_text or not query_cache_path_text:
    raise RuntimeError("Notebook 09 manifest is missing the selected candidate or query-cache path.")
CANDIDATE_POOL_PATH = Path(candidate_path_text)
QUERY_CACHE_PARQUET = Path(query_cache_path_text)
if not CANDIDATE_POOL_PATH.exists() or not QUERY_CACHE_PARQUET.exists():
    raise FileNotFoundError(f"Missing Notebook 09 dependency: {CANDIDATE_POOL_PATH}, {QUERY_CACHE_PARQUET}")

raw_candidates = pd.read_parquet(CANDIDATE_POOL_PATH)
require_columns(raw_candidates, REQUIRED_NOTEBOOK09_COLUMNS, "Notebook 09 winner pool")
if set(raw_candidates["category_id"].astype(str)) != {CATEGORY_ID}:
    raise RuntimeError("Candidate category_id mismatch.")
if set(raw_candidates["candidate_pool_role"].astype(str)) != {CANDIDATE_POOL_ROLE}:
    raise RuntimeError("Candidate role mismatch.")
if set(raw_candidates["method_slug"].astype(str)) != {CANDIDATE_RETRIEVAL_METHOD_KEY}:
    raise RuntimeError("Candidate method_slug mismatch.")
if set(raw_candidates["retrieval_method"].astype(str)) != {CANDIDATE_RETRIEVAL_METHOD_LABEL}:
    raise RuntimeError("Candidate retrieval_method mismatch.")
if not raw_candidates["target_parent_asin"].astype(str).eq(raw_candidates["gt_item_id"].astype(str)).all():
    raise RuntimeError("target_parent_asin and gt_item_id disagree.")

candidate_df = pd.DataFrame({
    "case_id": raw_candidates["case_id"].astype(str),
    "query_id": raw_candidates["query_id"].astype(str),
    "user_id": raw_candidates["user_id"].fillna("").astype(str),
    "regime": raw_candidates["regime"].map(normalize_regime),
    "sampling_bracket": raw_candidates["sampling_bracket"].fillna("").astype(str),
    "target_selection_mode": raw_candidates["target_selection_mode"].fillna("").astype(str),
    "query_method": STAGE1_QUERY_METHOD,
    "query_text": raw_candidates["query_text"].fillna("").astype(str),
    "target_item_id": raw_candidates["target_parent_asin"].astype(str),
    "gt_item_id": raw_candidates["gt_item_id"].astype(str),
    "target_timestamp_ms": to_timestamp_ms(raw_candidates["target_timestamp_ms"]).astype("int64"),
    "prior_review_n": pd.to_numeric(raw_candidates["prior_history_n"], errors="raise").astype(int),
    "prior_item_n": pd.to_numeric(raw_candidates["prior_history_n"], errors="raise").astype(int),
    "candidate_pool_role": raw_candidates["candidate_pool_role"].astype(str),
    "candidate_method_slug": raw_candidates["method_slug"].astype(str),
    "candidate_pool_type": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "retrieval_method": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "retrieval_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "item_id": raw_candidates["candidate_parent_asin"].astype(str),
    "rank": pd.to_numeric(raw_candidates["candidate_rank"], errors="raise").astype(int),
    "score": pd.to_numeric(raw_candidates["candidate_score"], errors="raise").astype(float),
    "candidate_score_source": raw_candidates["candidate_score_source"].fillna("").astype(str),
    "is_target": as_bool_series(raw_candidates["is_gt"]),
})
if candidate_df["regime"].eq("").any():
    raise RuntimeError("Notebook 09 contains an unknown or empty regime.")
if candidate_df["query_text"].str.strip().eq("").any():
    raise RuntimeError("Notebook 09 contains empty query text.")
if not np.isfinite(candidate_df["score"].to_numpy()).all():
    raise RuntimeError("Notebook 09 contains non-finite candidate scores.")
if candidate_df.duplicated(["query_id", "item_id"]).any() or candidate_df.duplicated(["query_id", "rank"]).any():
    raise RuntimeError("Notebook 09 contains duplicate query-item or query-rank rows.")
computed_target = candidate_df["item_id"].eq(candidate_df["target_item_id"])
if not computed_target.eq(candidate_df["is_target"]).all():
    raise RuntimeError("Notebook 09 is_gt is inconsistent with candidate identity.")

manifest_budget_k = int(stage1_pool_manifest.get("candidate_budget_k", POOL_K))
expected_k = int(stage1_pool_manifest.get("effective_candidate_count_per_query", POOL_K))
if manifest_budget_k != POOL_K or expected_k != POOL_K:
    raise RuntimeError("Notebook 09 candidate depth differs from the configured depth.")
candidate_count_per_query = candidate_df.groupby("query_id").size()
rank_stats = candidate_df.groupby("query_id")["rank"].agg(["min", "max", "nunique"])
if not candidate_count_per_query.eq(expected_k).all():
    raise RuntimeError("Notebook 09 does not provide exact-K candidates for every query.")
if not ((rank_stats["min"] == 1).all() and (rank_stats["max"] == expected_k).all() and (rank_stats["nunique"] == expected_k).all()):
    raise RuntimeError("Notebook 09 candidate ranks are not exactly 1 through K.")

query_cache_raw = pd.read_parquet(QUERY_CACHE_PARQUET)
require_columns(query_cache_raw, ["case_id", "target_rank_desc"], "Notebook 06 query cache")
query_cache_meta_df = query_cache_raw.copy()
query_cache_meta_df["case_id"] = query_cache_meta_df["case_id"].astype(str)
if query_cache_meta_df["case_id"].duplicated().any():
    raise RuntimeError("Notebook 06 query cache must be unique by case_id.")
active_case_ids = set(candidate_df["case_id"])
query_cache_meta_df = query_cache_meta_df[query_cache_meta_df["case_id"].isin(active_case_ids)].copy()
if set(query_cache_meta_df["case_id"]) != active_case_ids:
    raise RuntimeError("Notebook 06 query cache does not cover every Notebook 09 case.")

if "brand_or_name_leak_flag" in query_cache_meta_df.columns:
    brand_terms_added_to_synthetic_query_n = int(as_bool_series(query_cache_meta_df["brand_or_name_leak_flag"]).sum())
    query_brand_leak_qc_source = "query_cache_brand_or_name_leak_flag"
else:
    required_query_safety_audit_cols = [
        "query_evidence_source",
        "query_generation_status",
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "item_metadata_evidence_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "rating_evidence_used",
        "sentiment_evidence_used",
        "insufficient_review_evidence",
    ]
    missing_query_safety_audit_cols = [
        col for col in required_query_safety_audit_cols
        if col not in query_cache_meta_df.columns
    ]
    if missing_query_safety_audit_cols:
        raise RuntimeError(
            "Notebook 06 query cache is missing brand/name leak QC and required "
            f"target-review-safe audit columns: {missing_query_safety_audit_cols}"
        )
    forbidden_query_fallback_cols = [
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "item_metadata_evidence_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "rating_evidence_used",
        "sentiment_evidence_used",
        "insufficient_review_evidence",
    ]
    target_review_safe_source = query_cache_meta_df["query_evidence_source"].astype(str).eq("target_review_safe_signals_only")
    target_review_safe_status = query_cache_meta_df["query_generation_status"].astype(str).eq("generated_from_review_safe_signals")
    no_forbidden_query_fallback = ~query_cache_meta_df[forbidden_query_fallback_cols].apply(as_bool_series).any(axis=1)
    inferred_safe = target_review_safe_source & target_review_safe_status & no_forbidden_query_fallback
    if not inferred_safe.all():
        bad_query_safety_rows = query_cache_meta_df.loc[
            ~inferred_safe,
            ["case_id"] + required_query_safety_audit_cols,
        ].head(10).to_dict(orient="records")
        raise RuntimeError(
            "Notebook 06 query cache lacks brand_or_name_leak_flag and does not prove "
            f"target-review-safe query generation for all active cases: {bad_query_safety_rows}"
        )
    brand_terms_added_to_synthetic_query_n = 0
    query_brand_leak_qc_source = "inferred_zero_from_target_review_safe_query_cache_audit"

if brand_terms_added_to_synthetic_query_n != 0:
    raise RuntimeError("Brand or item-name terms were added to a synthetic query.")
query_cache_schema_map = {
    "case_id": "case_id",
    "target_rank_desc": "target_rank_desc",
    "brand_or_name_leak_flag": "brand_or_name_leak_flag" if "brand_or_name_leak_flag" in query_cache_meta_df.columns else query_brand_leak_qc_source,
}

query_meta_df = candidate_df[[
    "case_id", "query_id", "user_id", "regime", "sampling_bracket", "target_selection_mode",
    "query_method", "query_text", "target_item_id", "gt_item_id", "target_timestamp_ms",
    "prior_review_n", "prior_item_n", "candidate_pool_type", "retrieval_method",
    "retrieval_method_label", "candidate_method_slug",
]].drop_duplicates("query_id").copy()
query_meta_df = query_meta_df.merge(
    query_cache_meta_df[["case_id", "target_rank_desc"]],
    on="case_id", how="left", validate="one_to_one",
)
query_meta_df["target_rank_desc"] = pd.to_numeric(query_meta_df["target_rank_desc"], errors="raise").astype("Int32")
query_meta_df["regime"] = pd.Categorical(query_meta_df["regime"], categories=REGIME_ORDER, ordered=True)
query_meta_df["reranking_method"] = RERANKING_METHOD

candidate_pool_type_actual = CANDIDATE_RETRIEVAL_METHOD_LABEL
retrieval_method_actual = CANDIDATE_RETRIEVAL_METHOD_LABEL
retrieval_method_label_actual = CANDIDATE_RETRIEVAL_METHOD_LABEL
EXPECTED_QUERY_COUNT = int(stage1_pool_manifest.get("query_count", query_meta_df["query_id"].nunique()))
EXPECTED_CANDIDATE_ROWS_TOP1000 = EXPECTED_QUERY_COUNT * expected_k
EXPECTED_REGIME_COUNTS = query_meta_df["regime"].astype(str).value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict()
if query_meta_df["query_id"].nunique() != EXPECTED_QUERY_COUNT or len(candidate_df) != EXPECTED_CANDIDATE_ROWS_TOP1000:
    raise RuntimeError("Notebook 09 manifest counts disagree with the loaded pool.")

n_queries = int(query_meta_df["query_id"].nunique())
n_candidates = int(len(candidate_df))
load_runtime_sec = time.perf_counter() - load_start
record_runtime_step("load_inputs", load_runtime_sec, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates)
candidate_source_qc = {
    "stage1_candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
    "candidate_pool_role": CANDIDATE_POOL_ROLE,
    "candidate_retrieval_method_key": CANDIDATE_RETRIEVAL_METHOD_KEY,
    "candidate_retrieval_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "candidate_pool_path": str(CANDIDATE_POOL_PATH),
    "query_cache_path": str(QUERY_CACHE_PARQUET),
    "candidate_budget_policy": stage1_pool_manifest.get("candidate_budget_policy"),
    "candidate_budget_k": expected_k,
    "query_count": n_queries,
    "candidate_rows": n_candidates,
    "brand_terms_added_to_synthetic_query_n": brand_terms_added_to_synthetic_query_n,
    "query_brand_leak_qc_source": query_brand_leak_qc_source,
    "query_brand_leak_row_count": int(brand_terms_added_to_synthetic_query_n),
}

print("Candidate role:", CANDIDATE_POOL_ROLE)
print("Candidate method:", CANDIDATE_RETRIEVAL_METHOD_KEY, "-", CANDIDATE_RETRIEVAL_METHOD_LABEL)
print("Candidate rows:", n_candidates, "Cases:", n_queries, "Exact K:", expected_k)
print("Synthetic-query brand leakage: PASS")


Candidate role: baseline_query_only
Candidate method: hybrid_dense_bm25 - Dense-BM25 Hybrid
Candidate rows: 1968000 Cases: 1968 Exact K: 1000
Synthetic-query brand leakage: PASS


In [10]:
# =========================================================
# C-lite Sample Count QC
# =========================================================
query_sample_qc_source_df = query_meta_df.drop_duplicates("query_id").copy()
STRICT_SAMPLE_COUNT_QC = bool(globals().get("STRICT_SAMPLE_COUNT_QC", True))
actual_query_count = int(query_sample_qc_source_df["query_id"].nunique())
actual_regime_counts = {
    regime: int((query_sample_qc_source_df["regime"].astype(str) == regime).sum())
    for regime in EXPECTED_REGIME_COUNTS
}

sample_count_qc_df = pd.DataFrame([
    {
        "expected_total": int(EXPECTED_QUERY_COUNT),
        "actual_total": actual_query_count,
        "expected_cold": int(EXPECTED_REGIME_COUNTS.get("cold", 0)),
        "actual_cold": int(actual_regime_counts.get("cold", 0)),
        "expected_weak": int(EXPECTED_REGIME_COUNTS.get("weak", 0)),
        "actual_weak": int(actual_regime_counts.get("weak", 0)),
        "expected_strong": int(EXPECTED_REGIME_COUNTS.get("strong", 0)),
        "actual_strong": int(actual_regime_counts.get("strong", 0)),
    }
])

sample_count_qc_df["status"] = np.where(
    (
        sample_count_qc_df["expected_total"].eq(sample_count_qc_df["actual_total"])
        & sample_count_qc_df["expected_cold"].eq(sample_count_qc_df["actual_cold"])
        & sample_count_qc_df["expected_weak"].eq(sample_count_qc_df["actual_weak"])
        & sample_count_qc_df["expected_strong"].eq(sample_count_qc_df["actual_strong"])
    ),
    "pass",
    "fail",
)

print("Expected total sample size:", EXPECTED_QUERY_COUNT)
print("Actual total sample size:", actual_query_count)
print("Expected regime counts:", EXPECTED_REGIME_COUNTS)
print("Actual regime counts:", actual_regime_counts)
display(sample_count_qc_df)

if STRICT_SAMPLE_COUNT_QC and not sample_count_qc_df["status"].eq("pass").all():
    raise RuntimeError("C-lite sample count QC failed. Check upstream candidate/query-cache outputs.")


Expected total sample size: 1968
Actual total sample size: 1968
Expected regime counts: {'cold': 656, 'weak': 656, 'strong': 656}
Actual regime counts: {'cold': 656, 'weak': 656, 'strong': 656}


,expected_total,actual_total,expected_cold,actual_cold,expected_weak,actual_weak,expected_strong,actual_strong,status
0,1968,1968,656,656,656,656,656,656,pass


## 5. Load Catalog Facets and Diagnostic History

In [11]:
# =========================================================
# Exact Notebook 02 common facet contract
# =========================================================
PROFILE_ROLES = [
    "brand",
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
]
ATTRIBUTE_FAMILIES = tuple(PROFILE_ROLES)

FAMILY_STRENGTH_WEIGHTS = {
    "brand": 0.60,
    "category_or_product_type": 1.50,
    "form_texture": 1.10,
    "ingredient_or_composition": 1.20,
    "need_benefit_concern": 1.25,
    "claim_constraint": 0.90,
    "target_context": 0.70,
    "sensory": 0.50,
}

SUPPLEMENTARY_FAMILY_PRIORITY = ["brand", "need_benefit_concern", "ingredient_or_composition"]
SUPPLEMENTARY_METRIC_COLS = PREFERENCE_DIAGNOSTIC_METRICS


def get_item_family_set(item_id, family):
    return item_family_values.get(str(item_id), {}).get(str(family), frozenset())


def weighted_similarity_to_target(ranked_item_ids, gt_item_id, families, family_weights=None, k=PREFERENCE_ALIGNMENT_K):
    ranked = [str(item_id) for item_id in list(ranked_item_ids)[:int(k)]]
    if family_weights is None:
        family_weights = {family: 1.0 for family in families}
    weighted_sum = 0.0
    weight_total = 0.0
    family_scores = {}
    for family in families:
        target_values = get_item_family_set(str(gt_item_id), family)
        per_item = [jaccard(get_item_family_set(item_id, family), target_values) for item_id in ranked]
        score = float(np.mean(per_item)) if per_item else np.nan
        family_scores[family] = score
        weight = float(family_weights.get(family, 0.0))
        if pd.notna(score) and weight > 0:
            weighted_sum += score * weight
            weight_total += weight
    return {
        "weighted_facet_overlap_at_5": weighted_sum / weight_total if weight_total else np.nan,
        "brand_match_at_5": family_scores.get("brand", np.nan),
        "concern_match_at_5": family_scores.get("need_benefit_concern", np.nan),
        "ingredient_match_at_5": family_scores.get("ingredient_or_composition", np.nan),
    }


print("Notebook 02 profile-safe facet roles:", PROFILE_ROLES)


Notebook 02 profile-safe facet roles: ['brand', 'category_or_product_type', 'form_texture', 'ingredient_or_composition', 'need_benefit_concern', 'claim_constraint', 'target_context', 'sensory']


In [12]:

# =========================================================
# Temporal Artifact Config and Sample QC Contract
# =========================================================
QUERY_VARIANT = "C_lite"
EXPECTED_QUERY_COUNT = globals().get("EXPECTED_QUERY_COUNT")
EXPECTED_CANDIDATE_ROWS_TOP1000 = globals().get("EXPECTED_CANDIDATE_ROWS_TOP1000")
EXPECTED_REGIME_COUNTS = globals().get("EXPECTED_REGIME_COUNTS", {})
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
STRICT_SAMPLE_COUNT_QC = True

TEMPORAL_FEATURE_VERSION = "temporal_recency_v2_compact_prequery_only"
SHANNON_FEATURE_REGISTRY_VERSION = "herbal_shannon_compact_components_v2"
USE_TEMPORAL_RECENCY_FEATURES = True
TEMPORAL_WINDOWS_DAYS = [30, 90, 180]
MS_PER_DAY = 1000 * 60 * 60 * 24
RECENCY_FEATURE_FILL_DAYS = np.float32(9999.0)

ITEM_REVIEW_TIME_INDEX_PATH = PROCESSED_ITEMS_DIR / "item_review_time_index_herbal.parquet"
ITEM_REVIEW_DAILY_COUNTS_PATH = PROCESSED_ITEMS_DIR / "item_review_daily_counts_herbal.parquet"
ENTITY_REVIEW_DAILY_COUNTS_PATH = PROCESSED_ITEMS_DIR / "entity_review_daily_counts_herbal.parquet"
ITEM_TEMPORAL_SUMMARY_PATH = PROCESSED_ITEMS_DIR / "item_temporal_summary_herbal.parquet"
TEMPORAL_ARTIFACT_MANIFEST_PATH = PROCESSED_ITEMS_DIR / "temporal_artifact_manifest_herbal.json"

# CODEX CHECK: Confirm these Notebook 04 temporal paths resolve within this category and contain strictly pre-query timestamps. Fix path assignments only if the local artifact layout differs.
TEMPORAL_ARTIFACT_PATHS = {
    "item_review_time_index": ITEM_REVIEW_TIME_INDEX_PATH,
    "item_review_daily_counts": ITEM_REVIEW_DAILY_COUNTS_PATH,
    "entity_review_daily_counts": ENTITY_REVIEW_DAILY_COUNTS_PATH,
    "item_temporal_summary": ITEM_TEMPORAL_SUMMARY_PATH,
    "temporal_artifact_manifest": TEMPORAL_ARTIFACT_MANIFEST_PATH,
}

SHANNON_TEMPORAL_RECENCY_WEIGHT = 0.0
# User-history temporal recency is disabled for the Base no-prior Shannon control.
COMMON_TEMPORAL_ITEM_FEATURE_COLS = [
    "item_prequery_review_count",
    "item_recent_review_count_30d",
    "item_recent_review_count_90d",
    "item_recent_review_count_180d",
    "item_recent_review_share_30d",
    "item_recent_review_share_90d",
    "item_recent_review_share_180d",
    "item_review_velocity_30d",
    "item_review_velocity_90d",
    "item_review_velocity_180d",
    "item_last_review_gap_days",
    "item_first_review_age_days",
]
COMMON_TEMPORAL_USER_FEATURE_COLS = []
COMMON_TEMPORAL_FAMILY_FEATURE_COLS = []
COMMON_TIME_FEATURE_COLS = COMMON_TEMPORAL_ITEM_FEATURE_COLS + COMMON_TEMPORAL_USER_FEATURE_COLS + COMMON_TEMPORAL_FAMILY_FEATURE_COLS

FORBIDDEN_MODEL_FEATURE_COLS = {
    "timestamp_ms",
    "target_timestamp_ms",
    "target_review_text",
    "review_text",
    "review_body",
    "raw_review_text",
}

print("QUERY_VARIANT:", QUERY_VARIANT)
print("EXPECTED_QUERY_COUNT:", EXPECTED_QUERY_COUNT)
print("EXPECTED_REGIME_COUNTS:", EXPECTED_REGIME_COUNTS)
print("TEMPORAL_FEATURE_VERSION:", TEMPORAL_FEATURE_VERSION)
print("SHANNON_TEMPORAL_RECENCY_WEIGHT:", SHANNON_TEMPORAL_RECENCY_WEIGHT)


QUERY_VARIANT: C_lite
EXPECTED_QUERY_COUNT: 1968
EXPECTED_REGIME_COUNTS: {'cold': 656, 'weak': 656, 'strong': 656}
TEMPORAL_FEATURE_VERSION: temporal_recency_v2_compact_prequery_only
SHANNON_TEMPORAL_RECENCY_WEIGHT: 0.0


In [13]:
# =========================================================
# Load exact Notebook 02 catalog/facets and Notebook 03 All Prior
# =========================================================
artifact_load_start = time.perf_counter()

schema_required = ["parent_asin", "brand_facet_text", "common_query_safe_facet_text"]
facet_required = ["parent_asin", "facet_role", "facet_value_norm", "is_brand", "is_review_derived", "is_profile_safe"]
item_schema_raw = pd.read_parquet(ITEM_SCHEMA_PARQUET, columns=schema_required)
items_facets_raw = pd.read_parquet(ITEMS_FACETS_PARQUET, columns=facet_required)
require_columns(item_schema_raw, schema_required, "Notebook 02 item schema")
require_columns(items_facets_raw, facet_required, "Notebook 02 common long facets")

item_schema_raw["item_id"] = item_schema_raw["parent_asin"].astype(str)
if item_schema_raw["item_id"].duplicated().any():
    raise RuntimeError("Notebook 02 item schema must be unique by parent_asin.")
item_schema_raw["brand_raw"] = item_schema_raw["brand_facet_text"].fillna("").astype(str)
item_schema_raw["brand_norm"] = item_schema_raw["brand_raw"].map(normalize_text)
item_brand_raw_map = item_schema_raw.set_index("item_id")["brand_raw"].to_dict()
item_brand_map = item_schema_raw.set_index("item_id")["brand_norm"].to_dict()

items_facets_raw["item_id"] = items_facets_raw["parent_asin"].astype(str)
items_facets_raw["facet_role"] = items_facets_raw["facet_role"].astype(str)
items_facets_raw["facet_value_norm"] = items_facets_raw["facet_value_norm"].fillna("").map(normalize_text)
unknown_roles = sorted(set(items_facets_raw["facet_role"]) - set(PROFILE_ROLES) - {"review_derived_signal"})
if unknown_roles:
    raise RuntimeError(f"Unexpected Notebook 02 facet roles: {unknown_roles}")
profile_facets = items_facets_raw[
    as_bool_series(items_facets_raw["is_profile_safe"])
    & ~as_bool_series(items_facets_raw["is_review_derived"])
    & items_facets_raw["facet_role"].isin(PROFILE_ROLES)
    & items_facets_raw["facet_value_norm"].ne("")
][["item_id", "facet_role", "facet_value_norm", "is_brand"]].drop_duplicates()
if as_bool_series(profile_facets.loc[profile_facets["facet_role"].ne("brand"), "is_brand"]).any():
    raise RuntimeError("Notebook 02 is_brand conflicts with facet_role.")

family_map = defaultdict(lambda: defaultdict(set))
for row in profile_facets.itertuples(index=False):
    family_map[str(row.item_id)][str(row.facet_role)].add(str(row.facet_value_norm))
for item_id, brand_value in item_brand_map.items():
    if brand_value:
        family_map[str(item_id)]["brand"].add(str(brand_value))
item_family_values = {
    item_id: {role: frozenset(values) for role, values in role_map.items() if values}
    for item_id, role_map in family_map.items()
}

candidate_item_ids = set(candidate_df["item_id"].astype(str))
missing_candidate_items = sorted(candidate_item_ids - set(item_brand_map))
if missing_candidate_items:
    raise RuntimeError(f"Notebook 02 item schema misses candidate items: n={len(missing_candidate_items)}")
candidate_df["candidate_brand_raw"] = candidate_df["item_id"].map(item_brand_raw_map).fillna("").astype(str)
candidate_df["candidate_brand_norm"] = candidate_df["item_id"].map(item_brand_map).fillna("").astype(str)
candidate_df["candidate_brand_present"] = candidate_df["candidate_brand_norm"].ne("").astype("int8")
candidate_brand_nonnull_rate = float(candidate_df["candidate_brand_present"].mean())

prior_history_schema_map = {}
prior_history_df = pd.DataFrame(columns=[
    "case_id", "query_id", "user_id", "prior_item_id", "prior_timestamp_ms", "target_timestamp_ms",
])
same_target_item_prior_rows = 0
temporal_validation_passed = True
if USE_USER_PRIOR_FEATURES:
    prior_required = ["case_id", "user_id", "target_parent_asin", "target_timestamp_ms", "prior_parent_asin", "prior_timestamp_ms"]
    prior_item_column = "prior_parent_asin"
    prior_raw = pd.read_parquet(PRIOR_HISTORY_PARQUET, columns=prior_required)
    require_columns(prior_raw, prior_required, "Notebook 03 All Prior")
    prior_history_schema_map = {column: column for column in prior_required}
    prior_std = pd.DataFrame({
        "case_id": prior_raw["case_id"].astype(str),
        "user_id": prior_raw["user_id"].fillna("").astype(str),
        "prior_item_id": prior_raw[prior_item_column].fillna("").astype(str),
        "prior_timestamp_ms": to_timestamp_ms(prior_raw["prior_timestamp_ms"]),
        "upstream_target_timestamp_ms": to_timestamp_ms(prior_raw["target_timestamp_ms"]),
    })
    if CATEGORY_ID == "herbal":
        prior_std["upstream_target_item_id"] = prior_raw["target_parent_asin"].astype(str)
    query_lookup = query_meta_df[["case_id", "query_id", "user_id", "target_item_id", "target_timestamp_ms"]].copy()
    prior_std = prior_std.merge(query_lookup, on=["case_id", "user_id"], how="inner", validate="many_to_one")
    if not prior_std["upstream_target_timestamp_ms"].astype("int64").eq(prior_std["target_timestamp_ms"].astype("int64")).all():
        raise RuntimeError("Notebook 03 and Notebook 09 target timestamps disagree.")
    if CATEGORY_ID == "herbal" and not prior_std["upstream_target_item_id"].eq(prior_std["target_item_id"]).all():
        raise RuntimeError("Notebook 03 and Notebook 09 target items disagree.")
    prior_std = prior_std[
        prior_std["user_id"].ne("") & prior_std["prior_item_id"].ne("") & prior_std["prior_timestamp_ms"].notna()
    ].copy()
    prior_std["prior_timestamp_ms"] = prior_std["prior_timestamp_ms"].astype("int64")
    temporal_validation_passed = bool(prior_std["prior_timestamp_ms"].lt(prior_std["target_timestamp_ms"]).all())
    if not temporal_validation_passed:
        raise RuntimeError("All Prior contains an interaction at or after the target timestamp.")
    same_target_item_prior_rows = int(prior_std["prior_item_id"].eq(prior_std["target_item_id"]).sum())
    if same_target_item_prior_rows != 0:
        raise RuntimeError("All Prior contains the held-out target parent item.")
    prior_history_df = prior_std[[
        "case_id", "query_id", "user_id", "prior_item_id", "prior_timestamp_ms", "target_timestamp_ms",
    ]].reset_index(drop=True)
    missing_prior_items = sorted(set(prior_history_df["prior_item_id"]) - set(item_brand_map))
    missing_prior_item_count = int(len(missing_prior_items))
    print("Prior-history items missing from Notebook 02 item schema:", missing_prior_item_count)
    if missing_prior_items:
        print("Missing prior item examples:", missing_prior_items[:10])
    actual_prior_count = prior_history_df.groupby("query_id").size().reindex(query_meta_df["query_id"], fill_value=0).astype(int)
    cold_by_query = query_meta_df.set_index("query_id")["regime"].astype(str).eq("cold")
    if (cold_by_query != actual_prior_count.eq(0)).any():
        raise RuntimeError("Cold regime disagrees with the exact All Prior event count.")

global_value_vocab = {family: set() for family in ATTRIBUTE_FAMILIES}
family_item_support = {family: 0 for family in ATTRIBUTE_FAMILIES}
for family_values in item_family_values.values():
    for family in ATTRIBUTE_FAMILIES:
        values = set(family_values.get(family, frozenset()))
        if values:
            family_item_support[family] += 1
            global_value_vocab[family].update(values)
raw_global_family_weight = {
    family: float(family_item_support[family]) * float(FAMILY_STRENGTH_WEIGHTS[family])
    for family in ATTRIBUTE_FAMILIES
}
global_weight_total = float(sum(raw_global_family_weight.values()))
global_family_weights = {
    family: raw_global_family_weight[family] / global_weight_total if global_weight_total else 0.0
    for family in ATTRIBUTE_FAMILIES
}
ITEM_FACET_SOURCE_QC = {
    "policy": ITEM_FACET_EVIDENCE_POLICY,
    "rows_total": int(len(items_facets_raw)),
    "rows_retained": int(len(profile_facets)),
    "review_derived_rows_excluded": int(as_bool_series(items_facets_raw["is_review_derived"]).sum()),
    "filter_source": "exact_notebook02_is_profile_safe_and_not_review_derived",
    "brand_column": "brand_facet_text",
}

artifact_load_runtime_sec = time.perf_counter() - artifact_load_start
record_runtime_step("load_item_and_history_artifacts", artifact_load_runtime_sec, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates)
print("Candidate brand non-null rate:", round(candidate_brand_nonnull_rate, 6))
print("All Prior events:", len(prior_history_df), "Temporal validation:", temporal_validation_passed)
print("Notebook 04 artifacts read directly:", bool(CATEGORY_ID == "herbal" and USE_TEMPORAL_RECENCY_FEATURES))


Candidate brand non-null rate: 0.997478
All Prior events: 0 Temporal validation: True
Notebook 04 artifacts read directly: True


In [14]:

# =========================================================
# Temporal Artifact Loading
# =========================================================
with runtime_step("load_inputs", pool_depth=RUNTIME_REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    if USE_TEMPORAL_RECENCY_FEATURES:
        missing_temporal_paths = {
            name: str(path)
            for name, path in TEMPORAL_ARTIFACT_PATHS.items()
            if name != "temporal_artifact_manifest" and not Path(path).exists()
        }
        if missing_temporal_paths:
            raise FileNotFoundError(
                "Missing temporal artifacts. Run 04_retrieval_artifact_herbal_temporal.ipynb first:\n"
                + json.dumps(missing_temporal_paths, indent=2)
            )

        item_review_time_index = pd.read_parquet(ITEM_REVIEW_TIME_INDEX_PATH)
        item_review_daily_counts = pd.read_parquet(ITEM_REVIEW_DAILY_COUNTS_PATH)
        entity_review_daily_counts = pd.read_parquet(ENTITY_REVIEW_DAILY_COUNTS_PATH)
        item_temporal_summary = pd.read_parquet(ITEM_TEMPORAL_SUMMARY_PATH)
        temporal_artifact_manifest = load_json_if_exists(TEMPORAL_ARTIFACT_MANIFEST_PATH) if "load_json_if_exists" in globals() else {}

        required_item_time_cols = ["parent_asin", "review_timestamp_ms"]
        missing_item_time_cols = [c for c in required_item_time_cols if c not in item_review_time_index.columns]
        if missing_item_time_cols:
            raise RuntimeError(f"item_review_time_index is missing required columns: {missing_item_time_cols}")

        item_review_time_index["parent_asin"] = item_review_time_index["parent_asin"].fillna("").astype(str).str.strip()
        item_review_time_index["review_timestamp_ms"] = pd.to_numeric(item_review_time_index["review_timestamp_ms"], errors="coerce")
        item_review_time_index = item_review_time_index.dropna(subset=["review_timestamp_ms"]).copy()
        item_review_time_index["review_timestamp_ms"] = item_review_time_index["review_timestamp_ms"].astype(np.int64)
        item_review_time_index = item_review_time_index[item_review_time_index["parent_asin"].ne("")].copy()

        if item_review_time_index.empty:
            raise RuntimeError("item_review_time_index is empty after timestamp cleaning.")

        print("Temporal artifacts loaded.")
        print("item_review_time_index rows:", len(item_review_time_index))
        print("item_review_daily_counts rows:", len(item_review_daily_counts))
        print("entity_review_daily_counts rows:", len(entity_review_daily_counts))
        print("item_temporal_summary rows:", len(item_temporal_summary))
    else:
        item_review_time_index = pd.DataFrame(columns=["parent_asin", "review_timestamp_ms"])
        item_review_daily_counts = pd.DataFrame()
        entity_review_daily_counts = pd.DataFrame()
        item_temporal_summary = pd.DataFrame()
        temporal_artifact_manifest = {}


Temporal artifacts loaded.
item_review_time_index rows: 590182
item_review_daily_counts rows: 401430
entity_review_daily_counts rows: 351874
item_temporal_summary rows: 19538


## 6. Enforce the No-Prior Identity Contract

In [15]:
# =========================================================
# Build diagnostic profile fields without activating them in the no-prior score
# =========================================================
def normalized_entropy(counter_obj, vocab_size):
    total = float(sum(counter_obj.values()))
    if total <= 0:
        return np.nan
    probabilities = np.asarray([value / total for value in counter_obj.values()], dtype=np.float64)
    entropy = float(-np.sum(probabilities * np.log(probabilities + 1e-12)))
    if vocab_size <= 1:
        return 0.0
    return float(np.clip(entropy / math.log(vocab_size), 0.0, 1.0))


FUNCTIONAL_FAMILIES = tuple(family for family in ATTRIBUTE_FAMILIES if family != "brand")
if not FUNCTIONAL_FAMILIES:
    raise RuntimeError("Shannon requires at least one non-brand functional family.")


def normalize_weight_dict(raw_dict):
    cleaned = {family: max(float(raw_dict.get(family, 0.0)), 0.0) for family in ATTRIBUTE_FAMILIES}
    total = float(sum(cleaned.values()))
    return {family: cleaned[family] / total if total else 0.0 for family in ATTRIBUTE_FAMILIES}


def history_factor(history_n):
    history_n = max(int(history_n), 0)
    return float(min(1.0, math.log1p(history_n) / math.log1p(SHANNON_HISTORY_SATURATION_N))) if history_n else 0.0


def compute_personalization_strength(regime, entropy_norm, history_n):
    if not USE_USER_PRIOR_FEATURES:
        return 0.0
    regime = normalize_regime(regime)
    if regime == "cold" or history_n <= 0 or pd.isna(entropy_norm):
        return 0.0
    concentration = float(np.clip(1.0 - float(entropy_norm), 0.0, 1.0))
    raw_strength = SHANNON_ALPHA * concentration * history_factor(history_n)
    return float(np.clip(raw_strength, 0.0, float(REGIME_STRENGTH_CAP[regime])))


def build_query_profiles(query_meta, prior_history):
    prior_items_by_query = prior_history.groupby("query_id", sort=False)["prior_item_id"].apply(list).to_dict() if len(prior_history) else {}
    rows = []
    profile_value_sets = {}
    profile_counters = {}
    for row in query_meta.itertuples(index=False):
        query_id = str(row.query_id)
        prior_items = [str(item_id) for item_id in prior_items_by_query.get(query_id, [])]
        unique_prior_items = set(prior_items)
        family_counters = {family: Counter() for family in ATTRIBUTE_FAMILIES}
        if USE_USER_PRIOR_FEATURES:
            # All Prior frequency is intentionally retained: repeated interactions are not deduplicated.
            for item_id in prior_items:
                for family in ATTRIBUTE_FAMILIES:
                    for value in get_item_family_set(item_id, family):
                        family_counters[family][value] += 1
        family_entropy = {
            family: normalized_entropy(family_counters[family], max(len(global_value_vocab[family]), 1))
            for family in ATTRIBUTE_FAMILIES
        }
        raw_weights = {
            family: math.log1p(sum(family_counters[family].values())) * float(FAMILY_STRENGTH_WEIGHTS[family])
            for family in ATTRIBUTE_FAMILIES
        }
        family_weights = normalize_weight_dict(raw_weights)
        # Keep brand outside the functional Shannon entropy gate. Brand is scored in its own compact block.
        entropy_terms = [family_entropy[family] for family in FUNCTIONAL_FAMILIES if pd.notna(family_entropy[family]) and family_weights[family] > 0]
        entropy_weights = [family_weights[family] for family in FUNCTIONAL_FAMILIES if pd.notna(family_entropy[family]) and family_weights[family] > 0]
        entropy_norm_profile = float(np.average(entropy_terms, weights=entropy_weights)) if entropy_terms else np.nan
        strength = compute_personalization_strength(str(row.regime), entropy_norm_profile, len(prior_items))
        value_sets = {family: frozenset(family_counters[family]) for family in ATTRIBUTE_FAMILIES}
        profile_value_sets[query_id] = value_sets
        profile_counters[query_id] = family_counters
        out_row = {
            "case_id": str(row.case_id),
            "query_id": query_id,
            "user_id": str(row.user_id),
            "regime": str(row.regime),
            "profile_history_review_n": int(len(prior_items)),
            "profile_history_item_n": int(len(unique_prior_items)),
            "user_entropy_norm_shannon": entropy_norm_profile,
            "profile_concentration": float(1.0 - entropy_norm_profile) if pd.notna(entropy_norm_profile) else np.nan,
            "history_factor": history_factor(len(prior_items)),
            "lambda_shannon": strength,
            "profile_values_json": json.dumps({family: sorted(value_sets[family]) for family in ATTRIBUTE_FAMILIES}, ensure_ascii=False),
        }
        if USE_USER_PRIOR_FEATURES:
            for family in ATTRIBUTE_FAMILIES:
                out_row[f"profile_weight__{family}"] = family_weights[family]
                out_row[f"profile_value_count__{family}"] = int(len(value_sets[family]))
                out_row[f"profile_entropy_norm__{family}"] = family_entropy[family]
        rows.append(out_row)
    return pd.DataFrame(rows), profile_value_sets, profile_counters


feature_start = time.perf_counter()
query_profiles_df, profile_value_sets_by_query, profile_counters_by_query = build_query_profiles(query_meta_df, prior_history_df)
candidate_identity_before = candidate_df[["query_id", "item_id"]].sort_values(["query_id", "item_id"]).reset_index(drop=True)
feature_base_df = candidate_df.merge(
    query_profiles_df.drop(columns=["user_id", "regime"], errors="ignore"),
    on=["case_id", "query_id"], how="left", validate="many_to_one",
)
candidate_identity_after = feature_base_df[["query_id", "item_id"]].sort_values(["query_id", "item_id"]).reset_index(drop=True)
if len(feature_base_df) != len(candidate_df) or not candidate_identity_before.equals(candidate_identity_after):
    raise RuntimeError("Feature construction changed candidate rows or candidate IDs.")

feature_base_df["profile_history_review_n"] = safe_int_series(feature_base_df["profile_history_review_n"], default=0)
feature_base_df["profile_history_item_n"] = safe_int_series(feature_base_df["profile_history_item_n"], default=0)
feature_base_df["lambda_shannon"] = safe_numeric(feature_base_df["lambda_shannon"], default=0.0).clip(0.0, 1.0).astype("float32")
feature_base_df["retrieval_score_norm_full_pool"] = minmax_by_group(feature_base_df, "query_id", "score", max_rank=POOL_K).astype("float32")

functional_personalization_score = np.zeros(len(feature_base_df), dtype=np.float32)
if USE_USER_PRIOR_FEATURES:
    query_ids = feature_base_df["query_id"].astype(str).to_numpy()
    item_ids = feature_base_df["item_id"].astype(str).to_numpy()
    profiles_by_query = query_profiles_df.set_index("query_id")
    for family in ATTRIBUTE_FAMILIES:
        weights_by_query = profiles_by_query[f"profile_weight__{family}"].to_dict()
        matches = np.fromiter(
            (jaccard(profile_value_sets_by_query[qid][family], get_item_family_set(item_id, family)) for qid, item_id in zip(query_ids, item_ids)),
            dtype=np.float32, count=len(feature_base_df),
        )
        weights = np.fromiter((float(weights_by_query[qid]) for qid in query_ids), dtype=np.float32, count=len(feature_base_df))
        feature_base_df[f"match__{family}"] = matches.astype("float32")
        feature_base_df[f"weight__{family}"] = weights.astype("float32")
        if family != "brand":
            functional_personalization_score += matches * weights
feature_base_df["functional_personalization_score"] = np.clip(
    functional_personalization_score, 0.0, 1.0
).astype("float32")
feature_base_df["brand_personalization_score"] = np.float32(0.0)
feature_base_df["personalization_score"] = feature_base_df["functional_personalization_score"]

USER_BRAND_AFFINITY_COLUMNS = [
    "candidate_brand_prior_interaction_count",
    "candidate_brand_prior_share",
    "candidate_brand_recency_weight",
    "user_prior_brand_entropy_norm",
]
if USE_USER_PRIOR_FEATURES:
    brand_events = prior_history_df.copy()
    brand_events["brand"] = brand_events["prior_item_id"].map(item_brand_map).fillna("").astype(str)
    brand_events = brand_events[brand_events["brand"].ne("")].copy()
    interaction_count = brand_events.groupby(["query_id", "brand"]).size().astype(int).to_dict()
    unique_item_count = brand_events.groupby(["query_id", "brand"])["prior_item_id"].nunique().astype(int).to_dict()
    brand_event_denominator = brand_events.groupby("query_id").size().astype(int).to_dict()
    brand_last_timestamp = brand_events.groupby(["query_id", "brand"])["prior_timestamp_ms"].max().astype("int64").to_dict()
    unique_brand_count = brand_events.groupby("query_id")["brand"].nunique().astype(int).to_dict()
    brand_entropy_norm = {}
    for query_id, group in brand_events.groupby("query_id", sort=False):
        entropy_value = normalized_entropy(
            Counter(group["brand"].astype(str)),
            max(len(global_value_vocab.get("brand", frozenset())), 1),
        )
        brand_entropy_norm[str(query_id)] = 0.0 if pd.isna(entropy_value) else float(entropy_value)
    dominant_brand = {}
    for query_id, group in brand_events.groupby("query_id", sort=False):
        counts = group["brand"].value_counts()
        max_count = int(counts.max())
        dominant_brand[str(query_id)] = sorted(counts[counts.eq(max_count)].index.astype(str))[0]
    keys = list(zip(feature_base_df["query_id"].astype(str), feature_base_df["candidate_brand_norm"].astype(str)))
    feature_base_df["candidate_brand_seen_in_prior"] = np.asarray([interaction_count.get(key, 0) > 0 for key in keys], dtype=np.int8)
    feature_base_df["candidate_brand_prior_interaction_count"] = np.asarray([interaction_count.get(key, 0) for key in keys], dtype=np.int32)
    feature_base_df["candidate_brand_prior_unique_item_count"] = np.asarray([unique_item_count.get(key, 0) for key in keys], dtype=np.int32)
    feature_base_df["candidate_brand_prior_share"] = np.asarray([
        interaction_count.get(key, 0) / max(brand_event_denominator.get(key[0], 0), 1) for key in keys
    ], dtype=np.float32)
    recency_values = []
    for key, target_timestamp in zip(keys, feature_base_df["target_timestamp_ms"]):
        last_timestamp = brand_last_timestamp.get(key)
        age_days = (int(target_timestamp) - int(last_timestamp)) / float(MS_PER_DAY) if last_timestamp is not None else np.inf
        recency_values.append(math.exp(-max(age_days, 0.0) / 180.0) if np.isfinite(age_days) else 0.0)
    feature_base_df["candidate_brand_recency_weight"] = np.asarray(recency_values, dtype=np.float32)
    feature_base_df["candidate_brand_is_dominant_prior_brand"] = np.asarray([
        bool(key[1]) and dominant_brand.get(key[0], "") == key[1] for key in keys
    ], dtype=np.int8)
    feature_base_df["user_prior_unique_brand_count"] = feature_base_df["query_id"].map(unique_brand_count).fillna(0).astype("int32")
    feature_base_df["user_prior_brand_entropy_norm"] = (
        feature_base_df["query_id"].map(brand_entropy_norm).fillna(0.0).clip(0.0, 1.0).astype("float32")
    )
    feature_base_df["candidate_brand_profile_weight"] = (
        feature_base_df["match__brand"] * feature_base_df["weight__brand"]
    ).astype("float32")
    brand_interaction_strength = (
        np.log1p(feature_base_df["candidate_brand_prior_interaction_count"].clip(lower=0))
        / np.log1p(feature_base_df["profile_history_review_n"].clip(lower=1))
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0).clip(0.0, 1.0)
    brand_loyalty = (1.0 - feature_base_df["user_prior_brand_entropy_norm"]).clip(0.0, 1.0)
    brand_component_raw = (
        (brand_interaction_strength
         + feature_base_df["candidate_brand_prior_share"].clip(0.0, 1.0)
         + feature_base_df["candidate_brand_recency_weight"].clip(0.0, 1.0)
         + brand_loyalty)
        / 4.0
    ) * feature_base_df["candidate_brand_seen_in_prior"].astype("float32")
    feature_base_df["brand_personalization_score"] = (
        brand_component_raw * feature_base_df["weight__brand"]
    ).clip(0.0, 1.0).astype("float32")
    feature_base_df["personalization_score"] = (
        feature_base_df["functional_personalization_score"]
        + feature_base_df["brand_personalization_score"]
    ).clip(0.0, 1.0).astype("float32")
    users_with_prior_brand = int(brand_events["query_id"].nunique())
    candidate_brand_prior_match_rate = float(feature_base_df["candidate_brand_seen_in_prior"].mean())
else:
    users_with_prior_brand = 0
    candidate_brand_prior_match_rate = 0.0

feature_base_df["shannon_feature_available"] = (
    feature_base_df["profile_history_review_n"].gt(0)
    & feature_base_df["personalization_score"].gt(0)
)

# The no-prior control is an exact Stage-1 no-op; candidate brand presence is not a scoring feature.
S2Q_FEATURE_COLUMNS = []
SHARED_ALL_PRIOR_FEATURE_COLUMNS = [
    "candidate_brand_present",
    *USER_BRAND_AFFINITY_COLUMNS,
    "lambda_shannon",
    "functional_personalization_score",
    "brand_personalization_score",
]
SHANNON_FEATURE_COLUMNS = SHARED_ALL_PRIOR_FEATURE_COLUMNS if USE_USER_PRIOR_FEATURES else S2Q_FEATURE_COLUMNS
missing_feature_columns = [column for column in SHANNON_FEATURE_COLUMNS if column not in feature_base_df.columns]
if missing_feature_columns:
    raise RuntimeError(f"Missing Shannon contract features: {missing_feature_columns}")
if len(SHANNON_FEATURE_COLUMNS) != len(set(SHANNON_FEATURE_COLUMNS)):
    raise RuntimeError("Duplicated Shannon feature columns.")
user_brand_feature_columns_S2Q = sorted(set(S2Q_FEATURE_COLUMNS) & set(USER_BRAND_AFFINITY_COLUMNS))
if user_brand_feature_columns_S2Q:
    raise RuntimeError("S2-Q includes user-brand affinity columns.")

feature_runtime_sec = time.perf_counter() - feature_start
record_runtime_step("feature_preparation", feature_runtime_sec, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates)
record_runtime_step("model_fit_or_tuning", 0.0, pool_depth=REPORT_POOL_DEPTH, n_queries=n_queries, n_candidates=n_candidates, note="deterministic heuristic; no training or tuning")
global_family_weights_df = pd.DataFrame([{
    "family": family,
    "family_strength_weight": FAMILY_STRENGTH_WEIGHTS[family],
    "item_support": family_item_support[family],
    "vocab_size": len(global_value_vocab[family]),
    "global_family_weight": global_family_weights[family],
    "mean_profile_weight": float(query_profiles_df[f"profile_weight__{family}"].mean()) if USE_USER_PRIOR_FEATURES else 0.0,
    "mean_match": float(feature_base_df[f"match__{family}"].mean()) if USE_USER_PRIOR_FEATURES else 0.0,
} for family in ATTRIBUTE_FAMILIES])

print("Candidate rows unchanged by feature construction: PASS")
print("Users with at least one prior brand:", users_with_prior_brand)
print("Candidate-brand prior-match rate:", round(candidate_brand_prior_match_rate, 6))


Candidate rows unchanged by feature construction: PASS
Users with at least one prior brand: 0
Candidate-brand prior-match rate: 0.0


In [16]:
# =========================================================
# Derive temporal diagnostics and fix the no-prior temporal score to zero
# =========================================================
def _days_between_ms(later_ms, earlier_ms):
    return (np.asarray(later_ms, dtype=np.float64) - np.asarray(earlier_ms, dtype=np.float64)) / float(MS_PER_DAY)


def build_item_timestamp_map(item_time_df: pd.DataFrame) -> dict:
    if item_time_df.empty:
        return {}
    work = item_time_df[["parent_asin", "review_timestamp_ms"]].copy()
    work["parent_asin"] = work["parent_asin"].fillna("").astype(str).str.strip()
    work["review_timestamp_ms"] = pd.to_numeric(work["review_timestamp_ms"], errors="coerce")
    work = work.dropna(subset=["review_timestamp_ms"])
    work = work[work["parent_asin"].ne("")]
    work["review_timestamp_ms"] = work["review_timestamp_ms"].astype(np.int64)
    out = {}
    for item_id, group in work.groupby("parent_asin", sort=False):
        ts = np.sort(group["review_timestamp_ms"].to_numpy(dtype=np.int64))
        if len(ts):
            out[str(item_id)] = ts
    return out


def add_item_temporal_features_shannon(df: pd.DataFrame, item_ts_map: dict) -> pd.DataFrame:
    n = len(df)
    prequery_count = np.zeros(n, dtype=np.float32)
    last_gap_days = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    first_age_days = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    recent_counts = {days: np.zeros(n, dtype=np.float32) for days in TEMPORAL_WINDOWS_DAYS}
    item_values = df["item_id"].astype(str).to_numpy()
    query_ts_values = pd.to_numeric(df["target_timestamp_ms"], errors="coerce").fillna(0).astype(np.int64).to_numpy()
    row_by_item = defaultdict(list)
    for idx, item_id in enumerate(item_values):
        row_by_item[str(item_id)].append(idx)
    for item_id, row_indices in tqdm(row_by_item.items(), desc="Build item temporal features", leave=False):
        ts = item_ts_map.get(str(item_id))
        if ts is None or len(ts) == 0:
            continue
        idx_arr = np.asarray(row_indices, dtype=np.int64)
        qts = query_ts_values[idx_arr]
        valid_q = qts > 0
        if not valid_q.any():
            continue
        idx_valid = idx_arr[valid_q]
        qts_valid = qts[valid_q]
        right = np.searchsorted(ts, qts_valid, side="left")
        prequery_count[idx_valid] = right.astype(np.float32)
        has_prior = right > 0
        if has_prior.any():
            idx_has = idx_valid[has_prior]
            qts_has = qts_valid[has_prior]
            right_has = right[has_prior]
            last_ts = ts[right_has - 1]
            first_ts = ts[0]
            last_gap_days[idx_has] = np.maximum(0.0, _days_between_ms(qts_has, last_ts)).astype(np.float32)
            first_age_days[idx_has] = np.maximum(0.0, _days_between_ms(qts_has, np.full_like(qts_has, first_ts))).astype(np.float32)
        for days in TEMPORAL_WINDOWS_DAYS:
            left = np.searchsorted(ts, qts_valid - int(days * MS_PER_DAY), side="left")
            recent_counts[days][idx_valid] = np.maximum(0, right - left).astype(np.float32)
    df["item_prequery_review_count"] = prequery_count
    df["item_last_review_gap_days"] = last_gap_days
    df["item_first_review_age_days"] = first_age_days
    denom = np.maximum(prequery_count, 1.0)
    for days in TEMPORAL_WINDOWS_DAYS:
        df[f"item_recent_review_count_{days}d"] = recent_counts[days].astype(np.float32)
        df[f"item_recent_review_share_{days}d"] = (recent_counts[days] / denom).astype(np.float32)
        df[f"item_review_velocity_{days}d"] = (recent_counts[days] / np.float32(days)).astype(np.float32)
    return df


def build_user_temporal_maps_shannon(prior_history: pd.DataFrame) -> dict:
    prior = prior_history.copy()
    if prior.empty:
        return {"user_last_ts": {}, "user_item_last_ts": {}, "user_item_recent_count_180d": {}, "user_facet_last_ts": {}, "user_facet_recent_count_180d": {}}
    required = ["case_id", "user_id", "prior_item_id", "prior_timestamp_ms", "target_timestamp_ms"]
    missing = [c for c in required if c not in prior.columns]
    if missing:
        raise RuntimeError(f"prior_history_df missing required temporal columns: {missing}")
    prior = prior[required].copy()
    prior["case_id"] = prior["case_id"].fillna("").astype(str)
    prior["user_id"] = prior["user_id"].fillna("").astype(str)
    prior["prior_item_id"] = prior["prior_item_id"].fillna("").astype(str)
    prior["prior_timestamp_ms"] = pd.to_numeric(prior["prior_timestamp_ms"], errors="coerce")
    prior["target_timestamp_ms"] = pd.to_numeric(prior["target_timestamp_ms"], errors="coerce")
    prior = prior.dropna(subset=["prior_timestamp_ms", "target_timestamp_ms"])
    prior["prior_timestamp_ms"] = prior["prior_timestamp_ms"].astype(np.int64)
    prior["target_timestamp_ms"] = prior["target_timestamp_ms"].astype(np.int64)
    prior = prior[
        prior["case_id"].ne("")
        & prior["user_id"].ne("")
        & prior["prior_item_id"].ne("")
        & prior["prior_timestamp_ms"].lt(prior["target_timestamp_ms"])
    ].copy()
    if prior.empty:
        return {"user_last_ts": {}, "user_item_last_ts": {}, "user_item_recent_count_180d": {}, "user_facet_last_ts": {}, "user_facet_recent_count_180d": {}}
    user_last_ts = prior.groupby(["case_id", "user_id"], sort=False)["prior_timestamp_ms"].max().to_dict()
    user_item_last_ts = prior.groupby(["case_id", "user_id", "prior_item_id"], sort=False)["prior_timestamp_ms"].max().to_dict()
    prior["event_age_days"] = (prior["target_timestamp_ms"].astype(np.float64) - prior["prior_timestamp_ms"].astype(np.float64)) / float(MS_PER_DAY)
    user_item_recent_count_180d = (
        prior.loc[prior["event_age_days"].le(180.0)]
        .groupby(["case_id", "user_id", "prior_item_id"], sort=False)
        .size()
        .astype(int)
        .to_dict()
    )
    rows = []
    for row in prior[["case_id", "user_id", "prior_item_id", "prior_timestamp_ms", "event_age_days"]].itertuples(index=False):
        family_map = item_family_values.get(str(row.prior_item_id), {})
        for family in ATTRIBUTE_FAMILIES:
            for label in family_map.get(family, frozenset()):
                label_norm = normalize_text(label)
                if label_norm:
                    rows.append((str(row.case_id), str(row.user_id), family, label_norm, int(row.prior_timestamp_ms), float(row.event_age_days)))
    if rows:
        pf = pd.DataFrame(rows, columns=["case_id", "user_id", "family", "label", "prior_timestamp_ms", "event_age_days"])
        user_facet_last_ts = pf.groupby(["case_id", "user_id", "family", "label"], sort=False)["prior_timestamp_ms"].max().to_dict()
        user_facet_recent_count_180d = (
            pf.loc[pf["event_age_days"].le(180.0)]
            .groupby(["case_id", "user_id", "family", "label"], sort=False)
            .size()
            .astype(int)
            .to_dict()
        )
    else:
        user_facet_last_ts = {}
        user_facet_recent_count_180d = {}
    return {
        "user_last_ts": user_last_ts,
        "user_item_last_ts": user_item_last_ts,
        "user_item_recent_count_180d": user_item_recent_count_180d,
        "user_facet_last_ts": user_facet_last_ts,
        "user_facet_recent_count_180d": user_facet_recent_count_180d,
    }


def add_user_temporal_features_shannon(df: pd.DataFrame, temporal_maps: dict) -> pd.DataFrame:
    n = len(df)
    user_last_gap = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    user_item_recency = np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32)
    user_item_recent_count = np.zeros(n, dtype=np.float32)
    family_recency = {family: np.full(n, RECENCY_FEATURE_FILL_DAYS, dtype=np.float32) for family in ATTRIBUTE_FAMILIES}
    family_recent_count = {family: np.zeros(n, dtype=np.float32) for family in ATTRIBUTE_FAMILIES}

    for idx, row in enumerate(tqdm(df[["case_id", "user_id", "item_id", "target_timestamp_ms"]].itertuples(index=False), total=n, desc="Build user temporal features", leave=False)):
        case_id = str(row.case_id)
        user_id = str(row.user_id)
        item_id = str(row.item_id)
        query_ts = int(row.target_timestamp_ms) if pd.notna(row.target_timestamp_ms) else 0
        if query_ts <= 0:
            continue
        user_key = (case_id, user_id)
        last_user_ts = temporal_maps["user_last_ts"].get(user_key)
        if last_user_ts is not None and int(last_user_ts) < query_ts:
            user_last_gap[idx] = max(0.0, float(query_ts - int(last_user_ts)) / float(MS_PER_DAY))
        exact_key = (case_id, user_id, item_id)
        last_item_ts = temporal_maps["user_item_last_ts"].get(exact_key)
        if last_item_ts is not None and int(last_item_ts) < query_ts:
            user_item_recency[idx] = max(0.0, float(query_ts - int(last_item_ts)) / float(MS_PER_DAY))
        user_item_recent_count[idx] = float(temporal_maps["user_item_recent_count_180d"].get(exact_key, 0))
        item_family_map = item_family_values.get(item_id, {})
        for family in ATTRIBUTE_FAMILIES:
            labels = [normalize_text(v) for v in item_family_map.get(family, frozenset()) if normalize_text(v)]
            if not labels:
                continue
            last_ts_values = [
                int(ts)
                for label in labels
                for ts in [temporal_maps["user_facet_last_ts"].get((case_id, user_id, family, label))]
                if ts is not None and int(ts) < query_ts
            ]
            if last_ts_values:
                family_recency[family][idx] = max(0.0, float(query_ts - max(last_ts_values)) / float(MS_PER_DAY))
            family_recent_count[family][idx] = float(sum(int(temporal_maps["user_facet_recent_count_180d"].get((case_id, user_id, family, label), 0)) for label in labels))
    df["user_last_interaction_gap_days"] = user_last_gap
    df["user_item_recency_days"] = user_item_recency
    df["user_item_recent_count_180d"] = user_item_recent_count
    for family in ATTRIBUTE_FAMILIES:
        df[f"user_{family}_recency_days"] = family_recency[family]
        df[f"user_{family}_recent_count_180d"] = family_recent_count[family]
    return df


def minmax_series(values: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    min_v = float(numeric.min())
    max_v = float(numeric.max())
    if max_v <= min_v:
        return pd.Series(np.zeros(len(numeric), dtype=np.float32), index=numeric.index)
    return ((numeric - min_v) / (max_v - min_v)).astype(np.float32)


def inverse_gap_score(days_series: pd.Series, scale_days=90.0) -> pd.Series:
    days = pd.to_numeric(days_series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(RECENCY_FEATURE_FILL_DAYS)
    return (1.0 / (1.0 + (days / float(scale_days)))).clip(0.0, 1.0).astype(np.float32)


with runtime_step("feature_preparation", pool_depth=REPORT_POOL_DEPTH, runtime_measurement_type="wall_clock_step", derived_from_existing_runtime=False):
    if "target_timestamp_ms" not in feature_base_df.columns:
        if "target_timestamp_ms" in query_meta_df.columns:
            feature_base_df = feature_base_df.merge(query_meta_df[["query_id", "target_timestamp_ms"]].drop_duplicates("query_id"), on="query_id", how="left", validate="many_to_one")
        elif "timestamp_ms" in query_meta_df.columns:
            feature_base_df = feature_base_df.merge(query_meta_df[["query_id", "timestamp_ms"]].rename(columns={"timestamp_ms": "target_timestamp_ms"}).drop_duplicates("query_id"), on="query_id", how="left", validate="many_to_one")
    feature_base_df["target_timestamp_ms"] = pd.to_numeric(feature_base_df["target_timestamp_ms"], errors="coerce").fillna(0).astype(np.int64)

    item_timestamp_map = build_item_timestamp_map(item_review_time_index)
    user_temporal_maps = {"user_last_ts": {}, "user_item_last_ts": {}, "user_item_recent_count_180d": {}, "user_facet_last_ts": {}, "user_facet_recent_count_180d": {}}
    feature_base_df = add_item_temporal_features_shannon(feature_base_df, item_timestamp_map)
    # User-history temporal features and Shannon temporal score are disabled for the no-prior control.
    feature_base_df["shannon_temporal_item_score"] = np.float32(0.0)
    feature_base_df["shannon_temporal_user_score"] = np.float32(0.0)
    feature_base_df["shannon_temporal_recency_score"] = np.float32(0.0)

    temporal_feature_summary_df = pd.DataFrame({
        "feature": [col for col in COMMON_TIME_FEATURE_COLS if col in feature_base_df.columns] + ["shannon_temporal_recency_score"],
        "nonnull_rate": [float(feature_base_df[col].notna().mean()) for col in COMMON_TIME_FEATURE_COLS if col in feature_base_df.columns] + [float(feature_base_df["shannon_temporal_recency_score"].notna().mean())],
        "mean": [float(pd.to_numeric(feature_base_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).mean()) for col in COMMON_TIME_FEATURE_COLS if col in feature_base_df.columns] + [float(feature_base_df["shannon_temporal_recency_score"].mean())],
        "min": [float(pd.to_numeric(feature_base_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).min()) for col in COMMON_TIME_FEATURE_COLS if col in feature_base_df.columns] + [float(feature_base_df["shannon_temporal_recency_score"].min())],
        "max": [float(pd.to_numeric(feature_base_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).max()) for col in COMMON_TIME_FEATURE_COLS if col in feature_base_df.columns] + [float(feature_base_df["shannon_temporal_recency_score"].max())],
    })

    negative_temporal_cols = []
    for col in [c for c in COMMON_TIME_FEATURE_COLS if c in feature_base_df.columns and c.endswith("_days")]:
        if pd.to_numeric(feature_base_df[col], errors="coerce").fillna(0).lt(0).any():
            negative_temporal_cols.append(col)
    if negative_temporal_cols:
        raise RuntimeError(f"Negative temporal recency days detected: {negative_temporal_cols}")

    used_model_features_df = pd.DataFrame({
        "model": RERANKING_METHOD,
        "feature": ["rank", "retrieval_score_norm_pool", "personalization_score", "lambda_shannon", "shannon_temporal_recency_score"],
        "feature_group": ["baseline_rank_order", "retrieval_score_diagnostic", "diagnostic_user_profile", "diagnostic_profile_concentration", "diagnostic_temporal"],
        "used_for_final_ranking": [True, False, False, False, False],
        "used_for_final_score": [False, True, False, False, False],
        "diagnostic_only": [False, False, True, True, True],
        "is_temporal_feature": [False, False, False, False, True],
        "passes_no_prior_policy": [True, True, False, False, False],
        "no_prior_policy_passes": [True, True, False, False, False],
        "is_no_prior_excluded_feature": [False, False, True, True, True],
    })

    no_prior_model_feature_violations = used_model_features_df.loc[
        (used_model_features_df["used_for_final_ranking"] | used_model_features_df["used_for_final_score"])
        & ~used_model_features_df["no_prior_policy_passes"],
        "feature",
    ].astype(str).tolist()
    no_prior_score_components_disabled = bool(
        float(SHANNON_ALPHA) == 0.0
        and float(SHANNON_TEMPORAL_RECENCY_WEIGHT) == 0.0
        and float(pd.to_numeric(query_profiles_df["lambda_shannon"], errors="coerce").fillna(0.0).abs().max()) == 0.0
        and float(pd.to_numeric(feature_base_df["personalization_score"], errors="coerce").fillna(0.0).abs().max()) == 0.0
        and float(pd.to_numeric(feature_base_df["shannon_temporal_recency_score"], errors="coerce").fillna(0.0).abs().max()) == 0.0
    )

    used_model_features_df = pd.DataFrame({
        "model": RERANKING_METHOD,
        "feature": ["rank", "retrieval_score_norm_pool", "personalization_score", "lambda_shannon", "shannon_temporal_recency_score"],
        "feature_group": ["baseline_rank_order", "retrieval_score_diagnostic", "diagnostic_user_profile", "diagnostic_profile_concentration", "diagnostic_temporal"],
        "used_for_final_ranking": [True, False, False, False, False],
        "used_for_final_score": [False, True, False, False, False],
        "diagnostic_only": [False, False, True, True, True],
        "is_temporal_feature": [False, False, False, False, True],
        "passes_no_prior_policy": [True, True, False, False, False],
        "no_prior_policy_passes": [True, True, False, False, False],
        "is_no_prior_excluded_feature": [False, False, True, True, True],
    })

    no_prior_model_feature_violations = used_model_features_df.loc[
        (used_model_features_df["used_for_final_ranking"] | used_model_features_df["used_for_final_score"])
        & ~used_model_features_df["no_prior_policy_passes"],
        "feature",
    ].astype(str).tolist()
    no_prior_score_components_disabled = bool(
        float(SHANNON_ALPHA) == 0.0
        and float(SHANNON_TEMPORAL_RECENCY_WEIGHT) == 0.0
        and float(pd.to_numeric(query_profiles_df["lambda_shannon"], errors="coerce").fillna(0.0).abs().max()) == 0.0
        and float(pd.to_numeric(feature_base_df["personalization_score"], errors="coerce").fillna(0.0).abs().max()) == 0.0
        and float(pd.to_numeric(feature_base_df["shannon_temporal_recency_score"], errors="coerce").fillna(0.0).abs().max()) == 0.0
    )

    feature_leakage_qc_df = pd.DataFrame([
        {"check": "raw_timestamp_not_scored", "passed": True, "details": "timestamp columns are used only to derive pre-query item diagnostics."},
        {"check": "temporal_artifacts_loaded", "passed": bool(len(item_review_time_index) > 0), "details": str(ITEM_REVIEW_TIME_INDEX_PATH)},
        {"check": "negative_recency_days_absent", "passed": len(negative_temporal_cols) == 0, "details": json.dumps(negative_temporal_cols)},
        {"check": "no_prior_model_feature_contract_clean", "passed": len(no_prior_model_feature_violations) == 0, "details": json.dumps(no_prior_model_feature_violations)},
        {"check": "no_prior_score_components_disabled", "passed": no_prior_score_components_disabled, "details": NO_PRIOR_SCORE_POLICY},
    ])
    if not feature_leakage_qc_df["passed"].all():
        display(feature_leakage_qc_df)
        raise RuntimeError("No-prior Shannon score-component QC failed.")
    if no_prior_model_feature_violations:
        raise RuntimeError(f"No-prior Shannon score/ranking component violation: {no_prior_model_feature_violations}")

    feature_manifest = {
        "notebook_name": NOTEBOOK_NAME,
        "stage": STAGE,
        "category_id": CATEGORY_ID,
        "category_folder": CATEGORY_FOLDER,
        "reranking_method": RERANKING_METHOD,
        "experiment_condition": EXPERIMENT_CONDITION,
        "no_prior_feature_policy": NO_PRIOR_FEATURE_POLICY,
        "no_prior_shannon_manifest_note": NO_PRIOR_SHANNON_MANIFEST_NOTE,
        "no_prior_score_policy": NO_PRIOR_SCORE_POLICY,
        "no_prior_model_feature_contract_clean": True,
        "candidate_pool_type": candidate_pool_type_actual,
        "retrieval_method": retrieval_method_actual,
        "temporal_feature_version": TEMPORAL_FEATURE_VERSION,
        "raw_timestamp_features_used": False,
        "timestamp_metadata_columns": ["target_timestamp_ms", "timestamp_ms"],
        "strict_prequery_rule": "review_timestamp_ms < target_timestamp_ms and prior_timestamp_ms < target_timestamp_ms",
        "expected_query_count": None if EXPECTED_QUERY_COUNT is None else int(EXPECTED_QUERY_COUNT),
        "expected_regime_counts": EXPECTED_REGIME_COUNTS,
        "shannon_temporal_recency_weight": float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
        "shannon_alpha": float(SHANNON_ALPHA),
        "lambda_shannon_max": float(feature_base_df["lambda_shannon"].max()),
        "score_components_used_for_final_rank": ["rank"],
        "score_components_used_for_final_score": ["retrieval_score_norm_pool"],
        "used_model_features_validation_column": "no_prior_policy_passes",
        "filter_applied_to": ["final_score_formula", "final_ranking_order", "score_component_qc", "exported_feature_manifest"],
        "personalization_score_used_for_final_score": False,
        "shannon_temporal_recency_used_for_final_score": False,
        "temporal_feature_cols": [c for c in COMMON_TIME_FEATURE_COLS if c in feature_base_df.columns],
        "temporal_artifact_paths": {name: str(path) for name, path in TEMPORAL_ARTIFACT_PATHS.items()},
    }

    print("Temporal feature columns added:", len([c for c in COMMON_TIME_FEATURE_COLS if c in feature_base_df.columns]))
    print("Mean Shannon temporal recency score:", round(float(feature_base_df["shannon_temporal_recency_score"].mean()), 6))
    display(temporal_feature_summary_df.head(30))
    display(feature_leakage_qc_df)


# =========================================================
# Executed Shannon score/feature contract and RankP / Full parity
# =========================================================
feature_base_df["shannon_temporal_recency_score"] = safe_numeric(
    feature_base_df["shannon_temporal_recency_score"], default=0.0
).clip(0.0, 1.0).astype("float32")

SHANNON_SCORING_COMPONENTS = (
    ["retrieval_score_norm_pool"]
    if not USE_USER_PRIOR_FEATURES
    else ["retrieval_score_norm_pool", "lambda_shannon", "personalization_score", "shannon_temporal_recency_score"]
)
if USE_USER_PRIOR_FEATURES and "shannon_temporal_recency_score" not in SHANNON_FEATURE_COLUMNS:
    SHANNON_FEATURE_COLUMNS = [*SHANNON_FEATURE_COLUMNS, "shannon_temporal_recency_score"]

SHANNON_FEATURE_DTYPES = {column: str(feature_base_df[column].dtype) for column in SHANNON_FEATURE_COLUMNS}
SHARED_ALL_PRIOR_DIR = PROJECT_ROOT / "outputs" / "stage2_nonpersonalized_rerank" / "shannon"
SHARED_ALL_PRIOR_CONTRACT_PATH = SHARED_ALL_PRIOR_DIR / "shared_all_prior_shannon_contract.json"
shared_all_prior_contract = {
    "contract_version": "shared_all_prior_shannon_v2_compact_components",
    "category_id": CATEGORY_ID,
    "feature_columns": list(SHANNON_FEATURE_COLUMNS),
    "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
    "scoring_components": list(SHANNON_SCORING_COMPONENTS),
    "profile_roles": list(PROFILE_ROLES),
    "family_strength_weights": dict(FAMILY_STRENGTH_WEIGHTS),
    "shannon_alpha": float(SHANNON_ALPHA),
    "history_saturation_n": int(SHANNON_HISTORY_SATURATION_N),
    "regime_strength_cap": dict(REGIME_STRENGTH_CAP),
    "temporal_recency_weight": float(SHANNON_TEMPORAL_RECENCY_WEIGHT),
    "score_formula": "(1-lambda)*retrieval_norm + lambda*((1-recency_weight)*(functional_component+brand_component) + recency_weight*temporal_recency)",
    "tie_break": ["final_score desc", "retrieval_score_norm_pool desc", "candidate_rank asc", "candidate_item_id asc"],
    "missing_value_policy": "zero for absent profile match/count/share/recency; raw brand empty string",
    "cold_user_policy": "zero history-derived features and lambda_shannon=0; no fabricated preference",
    "history_source": "all_prior",
}
feature_columns_S2P = list(SHANNON_FEATURE_COLUMNS)
feature_columns_Full = feature_columns_S2P
feature_dtypes_S2P = dict(SHANNON_FEATURE_DTYPES)
feature_dtypes_Full = feature_dtypes_S2P
feature_schema_equality = feature_columns_S2P == feature_columns_Full and feature_dtypes_S2P == feature_dtypes_Full

if USE_USER_PRIOR_FEATURES and CANDIDATE_POOL_ROLE == "baseline_query_only":
    Path(SHARED_ALL_PRIOR_CONTRACT_PATH).write_text(
        json.dumps(make_jsonable(shared_all_prior_contract), ensure_ascii=False, indent=2), encoding="utf-8"
    )
elif USE_USER_PRIOR_FEATURES and CANDIDATE_POOL_ROLE == "personalized_retrieval":
    expected_shared_contract = load_json_required(SHARED_ALL_PRIOR_CONTRACT_PATH)
    comparable_keys = [
        "contract_version", "category_id", "feature_columns", "feature_dtypes", "scoring_components",
        "profile_roles", "family_strength_weights", "shannon_alpha", "history_saturation_n",
        "regime_strength_cap", "temporal_recency_weight", "score_formula", "tie_break",
        "missing_value_policy", "cold_user_policy", "history_source",
    ]
    mismatched_keys = [
        key for key in comparable_keys if expected_shared_contract.get(key) != shared_all_prior_contract.get(key)
    ]
    if mismatched_keys:
        raise RuntimeError(f"S2-P / Full Shannon contract mismatch: {mismatched_keys}")
    feature_columns_S2P = expected_shared_contract["feature_columns"]
    feature_columns_Full = SHANNON_FEATURE_COLUMNS
    feature_dtypes_S2P = expected_shared_contract["feature_dtypes"]
    feature_dtypes_Full = SHANNON_FEATURE_DTYPES
    feature_schema_equality = feature_columns_S2P == feature_columns_Full and feature_dtypes_S2P == feature_dtypes_Full
    if not feature_schema_equality:
        raise RuntimeError("feature_columns_S2P/Full or feature_dtypes_S2P/Full differ.")

raw_review_columns_in_model_input = sorted({
    column for column in SHANNON_FEATURE_COLUMNS
    if column in {"review_text", "review_body", "raw_review_text", "target_review_text", "heldout_review_text"}
})
if raw_review_columns_in_model_input:
    raise RuntimeError("Raw review columns entered the Shannon feature contract.")
if brand_terms_added_to_synthetic_query_n != 0 or same_target_item_prior_rows != 0 or not temporal_validation_passed:
    raise RuntimeError("A required leakage or temporal contract failed.")

used_model_features_df = pd.DataFrame([
    {
        "feature": feature,
        "feature_group": "direct_score_component" if feature in SHANNON_SCORING_COMPONENTS else "profile_component_or_audit",
        "is_common_model_feature": bool(USE_USER_PRIOR_FEATURES),
        "is_temporal_feature": feature == "shannon_temporal_recency_score",
        "is_dynamic_pool_feature": feature == "retrieval_score_norm_pool",
        "used_in_final_score": feature in SHANNON_SCORING_COMPONENTS or feature.startswith("match__") or feature.startswith("weight__") or feature == "candidate_brand_profile_weight",
    }
    for feature in list(dict.fromkeys([*SHANNON_SCORING_COMPONENTS, *SHANNON_FEATURE_COLUMNS]))
])

brand_contract = {
    "brand_query_enabled": False,
    "brand_candidate_visible": True,
    "brand_reranking_enabled": bool(USE_USER_PRIOR_FEATURES),
    "user_brand_affinity_enabled": bool(USE_USER_PRIOR_FEATURES),
    "history_source": "all_prior" if USE_USER_PRIOR_FEATURES else "none",
    "historical_population_review_signals_in_user_profile": False,
    "candidate_brand_source_column": "brand_facet_text",
    "candidate_brand_non_null_rate": float(candidate_brand_nonnull_rate),
    "notebook04_artifacts_read_directly": bool(CATEGORY_ID == "herbal" and USE_TEMPORAL_RECENCY_FEATURES),
}
contract_diagnostics = {
    "candidate_rows": int(len(feature_base_df)),
    "cases": int(feature_base_df["query_id"].nunique()),
    "candidate_brand_non_null_rate": float(candidate_brand_nonnull_rate),
    "users_with_at_least_one_prior_brand": int(users_with_prior_brand),
    "candidate_brand_prior_match_rate": float(candidate_brand_prior_match_rate),
    "cold_user_count": int(query_meta_df["regime"].astype(str).eq("cold").sum()),
    "s2q_feature_count": len(S2Q_FEATURE_COLUMNS),
    "s2p_feature_count": len(SHANNON_FEATURE_COLUMNS) if USE_USER_PRIOR_FEATURES else None,
    "full_feature_count": len(SHANNON_FEATURE_COLUMNS) if USE_USER_PRIOR_FEATURES else None,
    "brand_aware_feature_count": len(USER_BRAND_AFFINITY_COLUMNS) if USE_USER_PRIOR_FEATURES else 0,
    "s2p_full_feature_schema_equality": bool(feature_schema_equality) if USE_USER_PRIOR_FEATURES else None,
    "temporal_validation_status": "pass" if temporal_validation_passed else "fail",
}
feature_manifest.update({
    **brand_contract,
    "scoring_components": list(SHANNON_SCORING_COMPONENTS),
    "feature_columns": list(SHANNON_FEATURE_COLUMNS),
    "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
    "user_brand_feature_columns_s2q": list(user_brand_feature_columns_S2Q),
    "shared_all_prior_contract_path": str(SHARED_ALL_PRIOR_CONTRACT_PATH) if USE_USER_PRIOR_FEATURES else None,
    "contract_diagnostics": contract_diagnostics,
})

print("Shannon scoring components:", SHANNON_SCORING_COMPONENTS)
print("Feature contract columns:", len(SHANNON_FEATURE_COLUMNS))
print("RankP / Full feature-schema equality:", feature_schema_equality if USE_USER_PRIOR_FEATURES else "not applicable")
print("Temporal validation status:", contract_diagnostics["temporal_validation_status"])


Build item temporal features:   0%|          | 0/27236 [00:00<?, ?it/s]

Temporal feature columns added: 12
Mean Shannon temporal recency score: 0.0


,feature,nonnull_rate,mean,min,max
0,item_prequery_review_count,1.0,35.644211,0.000000,11601.000000
1,item_recent_review_count_30d,1.0,0.627698,0.000000,336.000000
2,item_recent_review_count_90d,1.0,1.874639,0.000000,885.000000
3,item_recent_review_count_180d,1.0,3.725515,0.000000,1684.000000
4,item_recent_review_share_30d,1.0,0.028392,0.000000,1.000000
5,item_recent_review_share_90d,1.0,0.075000,0.000000,1.000000
6,item_recent_review_share_180d,1.0,0.135485,0.000000,1.000000
7,item_review_velocity_30d,1.0,0.020923,0.000000,11.200000
8,item_review_velocity_90d,1.0,0.020829,0.000000,9.833333
9,item_review_velocity_180d,1.0,0.020697,0.000000,9.355556


,check,passed,details
0,raw_timestamp_not_scored,True,timestamp columns are used only to derive pre-...
1,temporal_artifacts_loaded,True,/content/drive/MyDrive/thesis_recsys/categorie...
2,negative_recency_days_absent,True,[]
3,no_prior_model_feature_contract_clean,True,[]
4,no_prior_score_components_disabled,True,"All user-profile, profile-concentration, and u..."


Shannon scoring components: ['retrieval_score_norm_pool']
Feature contract columns: 0
RankP / Full feature-schema equality: not applicable
Temporal validation status: pass


## 7. Copy the Frozen Pool and Evaluate

In [17]:
def score_rank_one_depth(feature_df, pool_depth, rerank_method):
    work = feature_df[feature_df['rank'].le(int(pool_depth))].copy()

    t_score = time.perf_counter()
    work['retrieval_score_norm_pool'] = minmax_by_group(work, 'query_id', 'score', max_rank=int(pool_depth))
    if rerank_method == BASELINE_RERANK_METHOD:
        work['final_score'] = work['retrieval_score_norm_pool']
        work['lambda_applied'] = 0.0
        work['personalization_score_scaled'] = 0.0
    elif rerank_method == SHANNON_OUTPUT_RERANK_METHOD:
        work['lambda_applied'] = 0.0
        work['personalization_score_scaled'] = 0.0
        work['temporal_recency_score_scaled'] = 0.0
        work['personalization_score_with_recency'] = 0.0
        work['final_score'] = work['retrieval_score_norm_pool']
    else:
        raise ValueError(f'Unknown rerank_method: {rerank_method}')
    scoring_runtime_sec = time.perf_counter() - t_score

    t_sort = time.perf_counter()
    if rerank_method in {BASELINE_RERANK_METHOD, SHANNON_OUTPUT_RERANK_METHOD}:
        work = work.sort_values(['query_id', 'rank', 'item_id'], ascending=[True, True, True]).copy()
    else:
        work = work.sort_values(
            ['query_id', 'final_score', 'retrieval_score_norm_pool', 'rank', 'item_id'],
            ascending=[True, False, False, True, True],
        ).copy()
    work['new_rank'] = work.groupby('query_id').cumcount() + 1
    work['rank_shift'] = work['rank'].astype(int) - work['new_rank'].astype(int)
    work['pool_depth'] = int(pool_depth)
    work['rerank_method'] = rerank_method
    sorting_runtime_sec = time.perf_counter() - t_sort
    return work, scoring_runtime_sec, sorting_runtime_sec


def per_query_metrics_for_ranked_frame(ranked_df, rank_col, rerank_method, pool_depth):
    base_cols = [
        'case_id', 'query_id', 'user_id', 'regime', 'sampling_bracket', 'target_selection_mode',
        'query_method', 'query_text', 'target_item_id', 'gt_item_id', 'prior_review_n', 'prior_item_n',
        'user_total_reviews', 'query_token_len', 'removed_token_count', 'candidate_pool_type', 'retrieval_method', 'retrieval_method_label',
    ]
    base_cols = [c for c in base_cols if c in query_meta_df.columns]
    base = query_meta_df[base_cols].drop_duplicates('query_id').copy()

    target_ranks = (
        ranked_df[ranked_df['is_target'].astype(bool)]
        .groupby('query_id')[rank_col]
        .min()
        .rename('gt_rank')
        .reset_index()
    )
    out = base.merge(target_ranks, on='query_id', how='left')
    out['pool_depth'] = int(pool_depth)
    out['rerank_method'] = rerank_method
    out['gt_in_pool'] = out['gt_rank'].notna().astype(int)

    metric_rows = [metrics_at_rank(rank, EVAL_KS) for rank in out['gt_rank'].tolist()]
    metrics_df = pd.DataFrame(metric_rows)
    out = pd.concat([out.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

    profile_keep = [
        'query_id', 'profile_history_review_n', 'profile_history_item_n', 'user_entropy_norm_shannon',
        'profile_concentration', 'history_factor', 'lambda_shannon',
    ]
    out = out.merge(query_profiles_df[profile_keep], on='query_id', how='left', validate='one_to_one')
    out['lambda_applied'] = out['lambda_shannon'].fillna(0.0) if rerank_method == SHANNON_OUTPUT_RERANK_METHOD else 0.0

    # Preference-alignment diagnostics at the user-facing top-5.
    top5 = ranked_df[ranked_df['new_rank'].le(PREFERENCE_ALIGNMENT_K)].sort_values(['query_id', 'new_rank'])
    top_items_by_query = top5.groupby('query_id')['item_id'].apply(list).to_dict()
    supplement_families = [f for f in SUPPLEMENTARY_FAMILY_PRIORITY if f in ATTRIBUTE_FAMILIES]
    supplement_weights = {f: global_family_weights.get(f, 0.0) for f in supplement_families}
    supplementary_rows = [
        weighted_similarity_to_target(
            top_items_by_query.get(str(row.query_id), []),
            str(row.gt_item_id),
            supplement_families,
            family_weights=supplement_weights,
            k=PREFERENCE_ALIGNMENT_K,
        )
        for row in out.itertuples(index=False)
    ]
    out = pd.concat([out.reset_index(drop=True), pd.DataFrame(supplementary_rows).reset_index(drop=True)], axis=1)
    return out


def summarize_metrics(df, group_cols):
    metric_cols = [c for c in df.columns if re.match(r'^(HitRate|MRR|NDCG)@\d+$', str(c))]
    agg_map = {'query_id': 'nunique'}
    for col in metric_cols + [c for c in SUPPLEMENTARY_METRIC_COLS if c in df.columns]:
        agg_map[col] = 'mean'
    out = df.groupby(group_cols, dropna=False).agg(agg_map).reset_index().rename(columns={'query_id': 'n_queries'})
    if 'regime' in out.columns:
        out = apply_regime_order(out, 'regime')
    sort_cols = [c for c in group_cols if c in out.columns]
    return out.sort_values(sort_cols).reset_index(drop=True)

print('Reranking and evaluation helpers ready.')


Reranking and evaluation helpers ready.


In [18]:
candidate_frames = []
per_query_frames = []
runtime_pool_depth_rows = []

for pool_depth in POOL_DEPTHS:
    depth_n_candidates = int(feature_base_df['rank'].le(int(pool_depth)).sum())

    for rerank_method in EXPECTED_RERANK_METHODS:
        ranked_df, scoring_runtime_sec, sorting_runtime_sec = score_rank_one_depth(feature_base_df, pool_depth, rerank_method)

        eval_start = time.perf_counter()
        per_query_df = per_query_metrics_for_ranked_frame(ranked_df, 'new_rank', rerank_method, pool_depth)
        eval_runtime_sec = time.perf_counter() - eval_start

        candidate_frames.append(ranked_df)
        per_query_frames.append(per_query_df)

        if rerank_method == SHANNON_OUTPUT_RERANK_METHOD:
            method_runtime_for_online = scoring_runtime_sec + sorting_runtime_sec
            runtime_pool_depth_rows.append({
                'category_id': CATEGORY_ID,
                'category_folder': CATEGORY_FOLDER,
                'method': RERANKING_METHOD,
                'reranker': rerank_method,
                'pool_depth': int(pool_depth),
                'n_queries': int(ranked_df['query_id'].nunique()),
                'n_candidates': int(len(ranked_df)),
                'model_scoring_runtime_sec': float(scoring_runtime_sec),
                'ranking_sorting_runtime_sec': float(sorting_runtime_sec),
                'rerank_runtime_sec': float(method_runtime_for_online),
                'rerank_runtime_sec_per_query': float(method_runtime_for_online / max(ranked_df['query_id'].nunique(), 1)),
                'rerank_runtime_sec_per_candidate': float(method_runtime_for_online / max(len(ranked_df), 1)),
                'eval_runtime_sec': float(eval_runtime_sec),
                'runtime_scope': 'stage2_online_reranking_excluding_stage1_retrieval',
                'candidate_pool_type': candidate_pool_type_actual,
                'retrieval_method': retrieval_method_actual,
                'retrieval_method_label': retrieval_method_label_actual,
                'reranking_method': RERANKING_METHOD,
            })

        # Preserve the current Notebook 10 wall-clock step rows for the report depth.
        if int(pool_depth) == REPORT_POOL_DEPTH and rerank_method == SHANNON_OUTPUT_RERANK_METHOD:
            record_runtime_step('model_scoring', scoring_runtime_sec, pool_depth=pool_depth, n_queries=n_queries, n_candidates=len(ranked_df))
            record_runtime_step('ranking_sorting', sorting_runtime_sec, pool_depth=pool_depth, n_queries=n_queries, n_candidates=len(ranked_df))
            record_runtime_step('evaluation', eval_runtime_sec, pool_depth=pool_depth, n_queries=n_queries, n_candidates=len(ranked_df))

reranked_candidates_all_depths_df = pd.concat(candidate_frames, ignore_index=True)
per_query_results_df = pd.concat(per_query_frames, ignore_index=True)
runtime_by_pool_depth_df = pd.DataFrame(runtime_pool_depth_rows)
if runtime_by_pool_depth_df.empty:
    raise RuntimeError("No per-depth runtime rows were recorded.")

runtime_by_pool_depth_df["pool_depth"] = runtime_by_pool_depth_df["pool_depth"].astype(int)
runtime_by_pool_depth_df["candidate_pool_depth"] = runtime_by_pool_depth_df["pool_depth"]

if "feature_preparation_runtime_sec" not in runtime_by_pool_depth_df.columns:
    runtime_by_pool_depth_df["feature_preparation_runtime_sec"] = 0.0
if "model_scoring_runtime_sec" not in runtime_by_pool_depth_df.columns:
    runtime_by_pool_depth_df["model_scoring_runtime_sec"] = 0.0
if "ranking_sorting_runtime_sec" not in runtime_by_pool_depth_df.columns:
    runtime_by_pool_depth_df["ranking_sorting_runtime_sec"] = 0.0
if "model_fit_or_tuning_runtime_sec" not in runtime_by_pool_depth_df.columns:
    if "training_runtime_sec" in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["model_fit_or_tuning_runtime_sec"] = runtime_by_pool_depth_df["training_runtime_sec"].astype(float)
    elif "fitting_runtime_sec" in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["model_fit_or_tuning_runtime_sec"] = runtime_by_pool_depth_df["fitting_runtime_sec"].astype(float)
    else:
        runtime_by_pool_depth_df["model_fit_or_tuning_runtime_sec"] = 0.0
if "online_operation_runtime_sec" not in runtime_by_pool_depth_df.columns:
    runtime_by_pool_depth_df["online_operation_runtime_sec"] = runtime_by_pool_depth_df["rerank_runtime_sec"].astype(float)
if "runtime_sec_per_query" not in runtime_by_pool_depth_df.columns:
    if "rerank_runtime_sec_per_query" in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["runtime_sec_per_query"] = runtime_by_pool_depth_df["rerank_runtime_sec_per_query"].astype(float)
    else:
        runtime_by_pool_depth_df["runtime_sec_per_query"] = runtime_by_pool_depth_df["online_operation_runtime_sec"] / runtime_by_pool_depth_df["n_queries"].clip(lower=1)
if "runtime_sec_per_candidate" not in runtime_by_pool_depth_df.columns:
    if "rerank_runtime_sec_per_candidate" in runtime_by_pool_depth_df.columns:
        runtime_by_pool_depth_df["runtime_sec_per_candidate"] = runtime_by_pool_depth_df["rerank_runtime_sec_per_candidate"].astype(float)
    else:
        runtime_by_pool_depth_df["runtime_sec_per_candidate"] = runtime_by_pool_depth_df["online_operation_runtime_sec"] / runtime_by_pool_depth_df["n_candidates"].clip(lower=1)
if "queries_per_second" not in runtime_by_pool_depth_df.columns:
    runtime_by_pool_depth_df["queries_per_second"] = runtime_by_pool_depth_df["n_queries"] / runtime_by_pool_depth_df["online_operation_runtime_sec"].replace(0, np.nan)
if "candidates_per_second" not in runtime_by_pool_depth_df.columns:
    runtime_by_pool_depth_df["candidates_per_second"] = runtime_by_pool_depth_df["n_candidates"] / runtime_by_pool_depth_df["online_operation_runtime_sec"].replace(0, np.nan)

expected_runtime_depths = {int(depth) for depth in POOL_DEPTHS}
observed_runtime_depths = set(runtime_by_pool_depth_df["pool_depth"].astype(int).unique())
if observed_runtime_depths != expected_runtime_depths:
    raise RuntimeError(
        "Per-depth runtime rows do not match evaluated pool depths: "
        f"expected {sorted(expected_runtime_depths)}, found {sorted(observed_runtime_depths)}"
    )


# Report-depth candidate export only, with both baseline and Shannon rows.
reranked_candidates_report_df = reranked_candidates_all_depths_df[reranked_candidates_all_depths_df['pool_depth'].eq(REPORT_POOL_DEPTH)].copy()
per_query_report_df = per_query_results_df[per_query_results_df['pool_depth'].eq(REPORT_POOL_DEPTH)].copy()

executed_methods = sorted(per_query_results_df['rerank_method'].astype(str).unique().tolist())
expected_pool_depths = sorted([int(x) for x in POOL_DEPTHS])
observed_pool_depths = sorted(pd.to_numeric(per_query_results_df['pool_depth'], errors='coerce').dropna().astype(int).unique().tolist())
if executed_methods != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError(f'Unexpected rerank methods: {executed_methods}')
if observed_pool_depths != expected_pool_depths:
    raise RuntimeError(f'Pool-depth outputs are incomplete. Expected {expected_pool_depths}, found {observed_pool_depths}')

# Candidate-set preservation check for full depth.
before_set = candidate_df.groupby('query_id')['item_id'].apply(lambda s: tuple(sorted(s.astype(str))))
after_set = (
    reranked_candidates_report_df[reranked_candidates_report_df['rerank_method'].eq(SHANNON_OUTPUT_RERANK_METHOD)]
    .groupby('query_id')['item_id']
    .apply(lambda s: tuple(sorted(s.astype(str))))
)
candidate_set_changed_by_reranker = not before_set.equals(after_set)
if candidate_set_changed_by_reranker:
    raise RuntimeError('Shannon reranker changed the report-depth candidate set.')


# No-prior Shannon is a no-op control, so its rank order should match the stage1 baseline.
baseline_order = (
    reranked_candidates_all_depths_df[reranked_candidates_all_depths_df['rerank_method'].eq(BASELINE_RERANK_METHOD)]
    .sort_values(['pool_depth', 'query_id', 'new_rank'], kind='mergesort')
    .groupby(['pool_depth', 'query_id'])['item_id']
    .apply(lambda values: tuple(values.astype(str)))
)
shannon_order = (
    reranked_candidates_all_depths_df[reranked_candidates_all_depths_df['rerank_method'].eq(SHANNON_OUTPUT_RERANK_METHOD)]
    .sort_values(['pool_depth', 'query_id', 'new_rank'], kind='mergesort')
    .groupby(['pool_depth', 'query_id'])['item_id']
    .apply(lambda values: tuple(values.astype(str)))
)
no_prior_shannon_ranking_matches_stage1_baseline = bool(baseline_order.equals(shannon_order))
if not no_prior_shannon_ranking_matches_stage1_baseline:
    mismatch_keys = [str(key) for key in baseline_order.index[baseline_order.ne(shannon_order.reindex(baseline_order.index))].tolist()[:10]]
    raise RuntimeError(f'No-prior Shannon ranking changed baseline order for pool/query keys: {mismatch_keys}')


print('per_query_results_df shape:', per_query_results_df.shape)
print('reranked_candidates_report_df shape:', reranked_candidates_report_df.shape)
print('runtime_by_pool_depth_df shape:', runtime_by_pool_depth_df.shape)
print('Executed methods:', executed_methods)
display(runtime_by_pool_depth_df.head(20))


per_query_results_df shape: (19680, 54)
reranked_candidates_report_df shape: (3936000, 63)
runtime_by_pool_depth_df shape: (5, 26)
Executed methods: ['shannon_no_prior_rerank', 'stage1_baseline']


,category_id,category_folder,method,reranker,pool_depth,n_queries,n_candidates,model_scoring_runtime_sec,ranking_sorting_runtime_sec,rerank_runtime_sec,rerank_runtime_sec_per_query,rerank_runtime_sec_per_candidate,eval_runtime_sec,runtime_scope,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,candidate_pool_depth,feature_preparation_runtime_sec,model_fit_or_tuning_runtime_sec,online_operation_runtime_sec,runtime_sec_per_query,runtime_sec_per_candidate,queries_per_second,candidates_per_second
0,herbal,herbal_supplements,shannon_no_prior,shannon_no_prior_rerank,100,1968,196800,0.047513,0.212155,0.259668,0.000132,0.000001,0.217160,stage2_online_reranking_excluding_stage1_retri...,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,100,0.0,0.0,0.259668,0.000132,0.000001,7578.907081,757890.708054
1,herbal,herbal_supplements,shannon_no_prior,shannon_no_prior_rerank,300,1968,590400,0.125068,0.645120,0.770188,0.000391,0.000001,0.213321,stage2_online_reranking_excluding_stage1_retri...,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,300,0.0,0.0,0.770188,0.000391,0.000001,2555.220891,766566.267303
2,herbal,herbal_supplements,shannon_no_prior,shannon_no_prior_rerank,500,1968,984000,0.202102,1.093087,1.295189,0.000658,0.000001,0.219962,stage2_online_reranking_excluding_stage1_retri...,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,500,0.0,0.0,1.295189,0.000658,0.000001,1519.469541,759734.770569
3,herbal,herbal_supplements,shannon_no_prior,shannon_no_prior_rerank,700,1968,1377600,0.284379,1.591641,1.876020,0.000953,0.000001,0.243718,stage2_online_reranking_excluding_stage1_retri...,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,700,0.0,0.0,1.876020,0.000953,0.000001,1049.029366,734320.555857
4,herbal,herbal_supplements,shannon_no_prior,shannon_no_prior_rerank,1000,1968,1968000,0.413424,2.444824,2.858248,0.001452,0.000001,0.233545,stage2_online_reranking_excluding_stage1_retri...,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,1000,0.0,0.0,2.858248,0.001452,0.000001,688.533783,688533.783095


## 8. Summarize Identity and Diagnostics

In [19]:
def add_summary_scope_metadata(df, summary_scope):
    work = df.copy()
    work['report_pool_depth'] = REPORT_POOL_DEPTH
    work['summary_scope'] = summary_scope
    work['condition'] = EXPERIMENT_CONDITION
    work['candidate_source'] = CANDIDATE_SOURCE_LABEL
    work['candidate_source_format'] = CANDIDATE_SOURCE_FORMAT
    work['candidate_pool_type'] = candidate_pool_type_actual
    work['retrieval_method'] = retrieval_method_actual
    work['retrieval_method_label'] = retrieval_method_label_actual
    work['reranking_method'] = RERANKING_METHOD
    work['comparison_set'] = 'native'
    work['category_id'] = CATEGORY_ID
    work['category_folder'] = CATEGORY_FOLDER
    work['category_label'] = CATEGORY_LABEL
    work['primary_metric'] = PRIMARY_STAGE2_METRIC
    return work


def build_uplift_summary(df, group_cols):
    group_cols = list(group_cols)
    ranking_metric_cols = [c for c in df.columns if re.match(r'^(HitRate|MRR|NDCG)@\d+$', str(c))]
    diagnostic_metric_cols = [c for c in UPLIFT_DIAGNOSTIC_METRICS if c in df.columns]
    metric_cols = ranking_metric_cols + diagnostic_metric_cols
    key_cols = group_cols + ['query_id']
    baseline = df[df['rerank_method'].astype(str).eq(BASELINE_RERANK_METHOD)][key_cols + metric_cols].copy()
    shannon = df[df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)][key_cols + metric_cols].copy()
    merged = shannon.merge(baseline, on=key_cols, how='inner', suffixes=('_shannon', '_baseline'), validate='one_to_one')
    for col in metric_cols:
        merged[f'delta_{col}'] = pd.to_numeric(merged[f'{col}_shannon'], errors='coerce').fillna(0.0) - pd.to_numeric(merged[f'{col}_baseline'], errors='coerce').fillna(0.0)
    delta_cols = [c for c in merged.columns if c.startswith('delta_')]
    if group_cols:
        out = merged.groupby(group_cols, dropna=False).agg({'query_id': 'nunique', **{c: 'mean' for c in delta_cols}}).reset_index().rename(columns={'query_id': 'n_queries'})
    else:
        out = pd.DataFrame([{**{'n_queries': int(merged['query_id'].nunique())}, **{c: float(merged[c].mean()) for c in delta_cols}}])
    return merged, out


summary_overall_df = add_summary_scope_metadata(summarize_metrics(per_query_report_df, ['rerank_method']), f'pool_depth_{REPORT_POOL_DEPTH}')
summary_by_pool_depth_df = add_summary_scope_metadata(summarize_metrics(per_query_results_df, ['pool_depth', 'rerank_method']), 'all_pool_depths')
summary_by_regime_df = add_summary_scope_metadata(summarize_metrics(per_query_report_df, ['regime', 'rerank_method']), f'pool_depth_{REPORT_POOL_DEPTH}')
summary_by_regime_pool_depth_df = add_summary_scope_metadata(summarize_metrics(per_query_results_df, ['regime', 'pool_depth', 'rerank_method']), 'all_pool_depths')

# Attach runtime to by-depth summary for quick efficiency inspection.
runtime_merge_cols = ['pool_depth', 'reranker', 'rerank_runtime_sec', 'rerank_runtime_sec_per_query', 'rerank_runtime_sec_per_candidate', 'eval_runtime_sec']
summary_by_pool_depth_df = summary_by_pool_depth_df.merge(
    runtime_by_pool_depth_df[runtime_merge_cols],
    left_on=['pool_depth', 'rerank_method'],
    right_on=['pool_depth', 'reranker'],
    how='left',
).drop(columns=['reranker'], errors='ignore')

overall_delta_detail_df, uplift_overall_df = build_uplift_summary(per_query_report_df, [])
pool_delta_detail_df, uplift_by_pool_depth_df = build_uplift_summary(per_query_results_df, ['pool_depth'])
regime_delta_detail_df, uplift_by_regime_df = build_uplift_summary(per_query_report_df, ['regime'])
regime_pool_delta_detail_df, uplift_by_regime_pool_depth_df = build_uplift_summary(per_query_results_df, ['regime', 'pool_depth'])

# Report-scope QC for downstream aggregation.
def pool_depth_value_string(df):
    if df is None or df.empty or 'pool_depth' not in df.columns:
        return ''
    vals = sorted(pd.to_numeric(df['pool_depth'], errors='coerce').dropna().astype(int).unique().tolist())
    return ','.join(map(str, vals))


def qc_row(output_file, expected_scope, source_dataframe, df):
    return {
        'notebook_name': NOTEBOOK_NAME,
        'output_file': output_file,
        'expected_scope': expected_scope,
        'actual_scope': expected_scope,
        'source_dataframe': source_dataframe,
        'pool_depth_values': pool_depth_value_string(df),
        'n_rows': int(len(df)) if df is not None else 0,
        'n_queries': int(df['query_id'].nunique()) if df is not None and 'query_id' in df.columns else int(pd.to_numeric(df['n_queries'], errors='coerce').max()) if df is not None and 'n_queries' in df.columns and len(df) else 0,
        'check_passed': True,
        'warning_message': '',
    }

report_summary_scope_qc_df = pd.DataFrame([
    qc_row('results_overall.csv', f'pool_depth_{REPORT_POOL_DEPTH}', 'per_query_report_df', per_query_report_df),
    qc_row('results_by_regime.csv', f'pool_depth_{REPORT_POOL_DEPTH}', 'per_query_report_df', per_query_report_df),
    qc_row('results_by_pool_depth.csv', 'all_pool_depths', 'per_query_results_df', per_query_results_df),
    qc_row('results_by_regime_pool_depth.csv', 'all_pool_depths', 'per_query_results_df', per_query_results_df),
    qc_row('per_query_metrics.parquet', 'all_pool_depths', 'per_query_results_df', per_query_results_df),
])

feature_diagnostics_df = pd.DataFrame([
    {
        'feature': col,
        'mean': float(pd.to_numeric(feature_base_df[col], errors='coerce').mean()),
        'std': float(pd.to_numeric(feature_base_df[col], errors='coerce').std()),
        'min': float(pd.to_numeric(feature_base_df[col], errors='coerce').min()),
        'max': float(pd.to_numeric(feature_base_df[col], errors='coerce').max()),
    }
    for col in ['retrieval_score_norm_full_pool', 'personalization_score', 'lambda_shannon'] + [f'match__{f}' for f in ATTRIBUTE_FAMILIES]
    if col in feature_base_df.columns
])

feature_importance_df = global_family_weights_df.rename(columns={'mean_profile_weight': 'importance'}).copy()
feature_importance_df['model_type'] = 'deterministic_shannon'
feature_importance_df['importance_source'] = 'mean_profile_family_weight'
feature_importance_summary_df = feature_importance_df[['family', 'importance', 'mean_match', 'global_family_weight', 'item_support', 'vocab_size', 'model_type', 'importance_source']].copy()

tuning_trials_df = pd.DataFrame([{
    'trial_id': 'deterministic_reference',
    'trial_source': 'fixed_heuristic',
    'selected': True,
    'offline_tuning_used': False,
    'shannon_alpha': SHANNON_ALPHA,
    'history_saturation_n': SHANNON_HISTORY_SATURATION_N,
    'cold_cap': REGIME_STRENGTH_CAP['cold'],
    'weak_cap': REGIME_STRENGTH_CAP['weak'],
    'strong_cap': REGIME_STRENGTH_CAP['strong'],
}])
tuning_summary = {
    'offline_tuning_used': False,
    'method_type': 'deterministic_shannon_entropy_heuristic',
    'selected_trial': make_jsonable(tuning_trials_df.iloc[0].to_dict()),
}

query_history_summary_df = pd.DataFrame([
    {
        'summary_name': 'query_profiles',
        'n_queries': int(query_profiles_df['query_id'].nunique()),
        'n_rows': int(len(query_profiles_df)),
        'profile_history_review_n_mean': float(query_profiles_df['profile_history_review_n'].mean()),
        'profile_history_item_n_mean': float(query_profiles_df['profile_history_item_n'].mean()),
    },
    {
        'summary_name': 'prior_history',
        'n_queries': int(prior_history_df['case_id'].nunique()),
        'n_rows': int(len(prior_history_df)),
        'profile_history_review_n_mean': float(prior_history_df.groupby('case_id').size().mean()) if len(prior_history_df) else 0.0,
        'profile_history_item_n_mean': float(prior_history_df.groupby('case_id')['prior_item_id'].nunique().mean()) if len(prior_history_df) else 0.0,
    },
])

print('Overall summary:')
display(summary_overall_df)
print('Uplift summary:')
display(uplift_overall_df)
print('By-depth summary preview:')
display(summary_by_pool_depth_df.head(20))


/tmp/ipykernel_8655/2949168301.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = df.groupby(group_cols, dropna=False).agg(agg_map).reset_index().rename(columns={'query_id': 'n_queries'})
/tmp/ipykernel_8655/2949168301.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = df.groupby(group_cols, dropna=False).agg(agg_map).reset_index().rename(columns={'query_id': 'n_queries'})


Overall summary:


/tmp/ipykernel_8655/2963330755.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = merged.groupby(group_cols, dropna=False).agg({'query_id': 'nunique', **{c: 'mean' for c in delta_cols}}).reset_index().rename(columns={'query_id': 'n_queries'})
/tmp/ipykernel_8655/2963330755.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = merged.groupby(group_cols, dropna=False).agg({'query_id': 'nunique', **{c: 'mean' for c in delta_cols}}).reset_index().rename(columns={'query_id': 'n_queries'})


,rerank_method,n_queries,HitRate@1,NDCG@1,MRR@1,HitRate@5,NDCG@5,MRR@5,HitRate@10,NDCG@10,MRR@10,HitRate@100,NDCG@100,MRR@100,HitRate@300,NDCG@300,MRR@300,HitRate@500,NDCG@500,MRR@500,HitRate@700,NDCG@700,MRR@700,HitRate@1000,NDCG@1000,MRR@1000,weighted_facet_overlap_at_5,brand_match_at_5,concern_match_at_5,ingredient_match_at_5,report_pool_depth,summary_scope,condition,candidate_source,candidate_source_format,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,comparison_set,category_id,category_folder,category_label,primary_metric
0,shannon_no_prior_rerank,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.34248,0.075153,0.026415,0.428862,0.085199,0.02664,0.499492,0.092867,0.02676,0.560976,0.099202,0.026834,0.024726,0.015244,0.04362,0.002122,1000,pool_depth_1000,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5
1,stage1_baseline,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.34248,0.075153,0.026415,0.428862,0.085199,0.02664,0.499492,0.092867,0.02676,0.560976,0.099202,0.026834,0.024726,0.015244,0.04362,0.002122,1000,pool_depth_1000,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5


Uplift summary:


,n_queries,delta_HitRate@1,delta_NDCG@1,delta_MRR@1,delta_HitRate@5,delta_NDCG@5,delta_MRR@5,delta_HitRate@10,delta_NDCG@10,delta_MRR@10,delta_HitRate@100,delta_NDCG@100,delta_MRR@100,delta_HitRate@300,delta_NDCG@300,delta_MRR@300,delta_HitRate@500,delta_NDCG@500,delta_MRR@500,delta_HitRate@700,delta_NDCG@700,delta_MRR@700,delta_HitRate@1000,delta_NDCG@1000,delta_MRR@1000,delta_weighted_facet_overlap_at_5,delta_brand_match_at_5,delta_concern_match_at_5,delta_ingredient_match_at_5
0,1968,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


By-depth summary preview:


,pool_depth,rerank_method,n_queries,HitRate@1,NDCG@1,MRR@1,HitRate@5,NDCG@5,MRR@5,HitRate@10,NDCG@10,MRR@10,HitRate@100,NDCG@100,MRR@100,HitRate@300,NDCG@300,MRR@300,HitRate@500,NDCG@500,MRR@500,HitRate@700,NDCG@700,MRR@700,HitRate@1000,NDCG@1000,MRR@1000,weighted_facet_overlap_at_5,brand_match_at_5,concern_match_at_5,ingredient_match_at_5,report_pool_depth,summary_scope,condition,candidate_source,candidate_source_format,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,comparison_set,category_id,category_folder,category_label,primary_metric,rerank_runtime_sec,rerank_runtime_sec_per_query,rerank_runtime_sec_per_candidate,eval_runtime_sec
0,100,shannon_no_prior_rerank,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.024726,0.015244,0.04362,0.002122,1000,all_pool_depths,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5,0.259668,0.000132,0.000001,0.217160
1,100,stage1_baseline,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.186484,0.054308,0.025509,0.024726,0.015244,0.04362,0.002122,1000,all_pool_depths,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5,NaN,NaN,NaN,NaN
2,300,shannon_no_prior_rerank,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.342480,0.075153,0.026415,0.342480,0.075153,0.026415,0.342480,0.075153,0.026415,0.342480,0.075153,0.026415,0.024726,0.015244,0.04362,0.002122,1000,all_pool_depths,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5,0.770188,0.000391,0.000001,0.213321
3,300,stage1_baseline,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.342480,0.075153,0.026415,0.342480,0.075153,0.026415,0.342480,0.075153,0.026415,0.342480,0.075153,0.026415,0.024726,0.015244,0.04362,0.002122,1000,all_pool_depths,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5,NaN,NaN,NaN,NaN
4,500,shannon_no_prior_rerank,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.342480,0.075153,0.026415,0.428862,0.085199,0.026640,0.428862,0.085199,0.026640,0.428862,0.085199,0.026640,0.024726,0.015244,0.04362,0.002122,1000,all_pool_depths,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Supplements,NDCG@5,1.295189,0.000658,0.000001,0.219962
5,500,stage1_baseline,1968,0.010671,0.010671,0.010671,0.037093,0.023847,0.019521,0.051321,0.028412,0.021382,0.186484,0.054308,0.025509,0.342480,0.075153,0.026415,0.428862,0.085199,0.026640,0.428862,0.085199,0.026640,0.428862,0.085199,0.026640,0.024726,0.015244,0.04362,0.002122,1000,all_pool_depths,s2q_no_prior_reranking,notebook09_baseline_query_only_hybrid_dense_bm25,notebook09_winner_long_pool,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,herbal,herbal_supplements,Herbal Suppleme

## 9. Export Base Outputs

In [20]:
# Canonical output paths expected by downstream Stage 2 comparison notebooks.

saved_paths = []

results_overall_path = OUT_DIR / 'results_overall.csv'
results_by_pool_depth_path = OUT_DIR / 'results_by_pool_depth.csv'
results_by_regime_path = OUT_DIR / 'results_by_regime.csv'
results_by_regime_pool_depth_path = OUT_DIR / 'results_by_regime_pool_depth.csv'
runtime_by_pool_depth_path = OUT_DIR / 'runtime_by_pool_depth.csv'
uplift_summary_path = OUT_DIR / 'uplift_summary.csv'
uplift_by_pool_depth_path = OUT_DIR / 'uplift_by_pool_depth.csv'
uplift_by_regime_path = OUT_DIR / 'uplift_by_regime.csv'
uplift_by_regime_pool_depth_path = OUT_DIR / 'uplift_by_regime_pool_depth.csv'
feature_table_path = OUT_DIR / 'feature_table.parquet'
per_query_results_path = OUT_DIR / 'per_query_metrics.parquet'
reranked_candidates_path = OUT_DIR / 'reranked_candidates.parquet'
query_profiles_path = OUT_DIR / 'user_profiles.parquet'
tuning_trials_path = OUT_DIR / 'tuning_trials.csv'
tuning_summary_json_path = OUT_DIR / 'tuning_summary.json'
query_history_summary_path = OUT_DIR / 'query_history_summary.csv'
global_family_weights_path = OUT_DIR / 'global_family_weights.csv'
feature_diagnostics_path = OUT_DIR / 'feature_diagnostics.csv'
feature_importance_path = OUT_DIR / 'feature_importance.csv'
feature_importance_summary_path = OUT_DIR / 'feature_importance_summary.csv'
config_snapshot_path = OUT_DIR / 'config_snapshot.json'
diagnostics_summary_path = OUT_DIR / 'diagnostics_summary.csv'
report_summary_scope_qc_path = OUT_DIR / 'report_summary_scope_qc.csv'
run_manifest_path = OUT_DIR / 'run_manifest.json'

export_material_start = time.perf_counter()

summary_overall_df.to_csv(results_overall_path, index=False)
summary_by_pool_depth_df.to_csv(results_by_pool_depth_path, index=False)
summary_by_regime_df.to_csv(results_by_regime_path, index=False)
summary_by_regime_pool_depth_df.to_csv(results_by_regime_pool_depth_path, index=False)
runtime_by_pool_depth_df.to_csv(runtime_by_pool_depth_path, index=False)
uplift_overall_df.to_csv(uplift_summary_path, index=False)
uplift_by_pool_depth_df.to_csv(uplift_by_pool_depth_path, index=False)
uplift_by_regime_df.to_csv(uplift_by_regime_path, index=False)
uplift_by_regime_pool_depth_df.to_csv(uplift_by_regime_pool_depth_path, index=False)

feature_export_cols = [
    "case_id", "query_id", "user_id", "regime", "item_id", "target_item_id", "gt_item_id",
    "rank", "score", "candidate_brand_raw", "retrieval_score_norm_full_pool",
    "profile_history_review_n", "profile_history_item_n", "user_entropy_norm_shannon",
    "profile_concentration", "shannon_feature_available",
] + list(SHANNON_FEATURE_COLUMNS)
feature_export_cols = list(dict.fromkeys(c for c in feature_export_cols if c in feature_base_df.columns))
feature_base_df[feature_export_cols].to_parquet(feature_table_path, index=False)

per_query_results_df.to_parquet(per_query_results_path, index=False)
reranked_candidates_report_df.to_parquet(reranked_candidates_path, index=False)
query_profiles_df.to_parquet(query_profiles_path, index=False)
tuning_trials_df.to_csv(tuning_trials_path, index=False)
query_history_summary_df.to_csv(query_history_summary_path, index=False)
global_family_weights_df.to_csv(global_family_weights_path, index=False)
feature_diagnostics_df.to_csv(feature_diagnostics_path, index=False)
feature_importance_df.to_csv(feature_importance_path, index=False)
feature_importance_summary_df.to_csv(feature_importance_summary_path, index=False)
report_summary_scope_qc_df.to_csv(report_summary_scope_qc_path, index=False)
with open(tuning_summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(make_jsonable(tuning_summary), f, ensure_ascii=False, indent=2)

config_snapshot = {
    'notebook_name': NOTEBOOK_NAME,
    'stage': STAGE,
    'category_id': CATEGORY_ID,
    'category_folder': CATEGORY_FOLDER,
    'category_label': CATEGORY_LABEL,
    'experiment_condition': EXPERIMENT_CONDITION,
    'condition': EXPERIMENT_CONDITION,
    'experiment_interpretation': 'baseline retrieval + no-prior Shannon control',
    'no_prior_manifest_note': NO_PRIOR_SCORE_POLICY,
    'no_prior_feature_policy': NO_PRIOR_FEATURE_POLICY,
    'no_prior_shannon_control_type': 'no_op_rank_preserving',
    'score_components_used_for_final_rank': ['rank'],
    'score_components_used_for_final_score': [],
    'candidate_source': CANDIDATE_SOURCE_LABEL,
    'query_brand_leak_qc_source': query_brand_leak_qc_source,
    'query_brand_leak_row_count': int(brand_terms_added_to_synthetic_query_n),
    'query_variant': QUERY_VARIANT,
    'candidate_source_format': CANDIDATE_SOURCE_FORMAT,
    'project_root': str(PROJECT_ROOT),
    'output_dir': str(OUT_DIR),
    'candidate_pool_path': str(CANDIDATE_POOL_PATH),
    'candidate_pool_type': candidate_pool_type_actual,
    'retrieval_method': retrieval_method_actual,
    'retrieval_method_label': retrieval_method_label_actual,
    'reranking_method': RERANKING_METHOD,
    'rerank_method': SHANNON_OUTPUT_RERANK_METHOD,
    'query_method': STAGE1_QUERY_METHOD,
    'pool_depths': [int(x) for x in POOL_DEPTHS],
    'report_pool_depth': int(REPORT_POOL_DEPTH),
    'eval_ks': [int(x) for x in EVAL_KS],
    'primary_stage2_metric': PRIMARY_STAGE2_METRIC,
    'primary_stage2_metrics': PRIMARY_STAGE2_METRICS,
    'primary_metrics': PRIMARY_STAGE2_METRICS,
    'secondary_stage2_metrics': SECONDARY_STAGE2_METRICS,
    'secondary_metrics': SECONDARY_STAGE2_METRICS,
    'supplementary_stage2_metrics': SUPPLEMENTARY_STAGE2_METRICS,
    'common_comparison_metrics': COMMON_STAGE2_COMPARISON_METRICS,
    'preference_diagnostic_metrics': PREFERENCE_DIAGNOSTIC_METRICS,
    'preference_diagnostic_families': PREFERENCE_DIAGNOSTIC_FAMILIES,
    'preference_diagnostic_family_weights': PREFERENCE_DIAGNOSTIC_FAMILY_WEIGHTS,
    'preference_alignment_k': int(PREFERENCE_ALIGNMENT_K),
    'expected_rerank_methods': list(EXPECTED_RERANK_METHODS),
    'deterministic_heuristic': True,
    'offline_tuning_used': False,

    'item_facet_evidence_policy': ITEM_FACET_EVIDENCE_POLICY,
    'item_facet_source_qc': ITEM_FACET_SOURCE_QC,
    'entropy_normalization_policy': ENTROPY_NORMALIZATION_POLICY,
    'shannon_scoring_policy': SHANNON_SCORING_POLICY,
    'algorithm_change_scope': ALGORITHM_CHANGE_SCOPE,
    'raw_reviews_loaded': False,
    'target_review_text_used_or_saved': False,
    'strict_prior_history_loaded': bool(USE_USER_PRIOR_FEATURES),
    'strict_prior_history_used_for_scoring': False,
    'prior_profile_columns_diagnostics_only': True,
    'same_item_prior_policy': 'not used for no-prior scoring; prior/profile columns are diagnostics/reporting only',
    'runtime_framework': 'wall_clock_step_plus_component_summary',
}
Path(config_snapshot_path).write_text(json.dumps(make_jsonable(config_snapshot), ensure_ascii=False, indent=2), encoding='utf-8')

saved_paths.extend([
    results_overall_path, results_by_pool_depth_path, results_by_regime_path, results_by_regime_pool_depth_path,
    runtime_by_pool_depth_path, uplift_summary_path, uplift_by_pool_depth_path, uplift_by_regime_path,
    uplift_by_regime_pool_depth_path, feature_table_path, per_query_results_path, reranked_candidates_path,
    query_profiles_path, tuning_trials_path, tuning_summary_json_path, query_history_summary_path,
    global_family_weights_path, feature_diagnostics_path, feature_importance_path, feature_importance_summary_path,
    report_summary_scope_qc_path, config_snapshot_path,
])

# Runtime export is called last so export_outputs and total_notebook include material-output writing.
export_material_runtime_sec = time.perf_counter() - export_material_start
export_n_candidates_report = int(len(reranked_candidates_report_df[reranked_candidates_report_df['rerank_method'].astype(str).eq(SHANNON_OUTPUT_RERANK_METHOD)]))
record_runtime_step(
    'export_outputs',
    export_material_runtime_sec,
    pool_depth=REPORT_POOL_DEPTH,
    n_queries=n_queries,
    n_candidates=export_n_candidates_report,
)
runtime_steps_df, runtime_notebook_summary_df, runtime_method_summary_at1000_df, runtime_pool_depth_diagnostic_df, runtime_method_components_at1000_df = export_runtime_logs(OUT_DIR, report_pool_depth=REPORT_POOL_DEPTH)
for runtime_path in [
    OUT_DIR / 'runtime_steps.csv',
    OUT_DIR / 'runtime_notebook_summary.csv',
    OUT_DIR / 'runtime_method_summary_at1000.csv',
    OUT_DIR / 'runtime_pool_depth_diagnostic.csv',
    OUT_DIR / 'runtime_method_components_at1000_herbal.csv',
]:
    saved_paths.append(runtime_path)

required_input_paths = {
    "stage1_candidate_pool_manifest": CANDIDATE_POOL_MANIFEST_PATH,
    "candidate_pool_path": CANDIDATE_POOL_PATH,
    "query_cache_parquet": QUERY_CACHE_PARQUET,
    "items_facets_parquet": ITEMS_FACETS_PARQUET,
}
if USE_USER_PRIOR_FEATURES:
    required_input_paths["prior_history_parquet"] = PRIOR_HISTORY_PARQUET

optional_input_paths = {
    "query_cache_summary_path": QUERY_CACHE_SUMMARY_PATH,
    "query_cache_config_path": QUERY_CACHE_CONFIG_PATH,
    "item_schema_parquet": ITEM_SCHEMA_PARQUET,
    "item_schema_base_parquet": ITEM_SCHEMA_BASE_PARQUET,
    "item_docs_parquet": ITEM_DOCS_PARQUET,
}

input_paths = {
    **{name: str(path) for name, path in required_input_paths.items()},
    **{name: str(path) for name, path in optional_input_paths.items()},
}

all_output_paths = saved_paths + [run_manifest_path, diagnostics_summary_path]

run_manifest = {
    **config_snapshot,
    'input_paths': input_paths,
    'output_paths': {Path(p).stem: str(p) for p in all_output_paths},
    'query_count': int(n_queries),
    'candidate_rows': int(n_candidates),
    'candidate_rows_report_depth': int(len(reranked_candidates_report_df)),
    'executed_rerank_methods': list(executed_methods),
    'candidate_set_changed_by_reranker': bool(candidate_set_changed_by_reranker),
    'regime_counts': query_meta_df['regime'].astype(str).value_counts().to_dict(),
    'created_outputs': [str(p) for p in all_output_paths],
}
Path(run_manifest_path).write_text(json.dumps(make_jsonable(run_manifest), ensure_ascii=False, indent=2), encoding='utf-8')
saved_paths.append(run_manifest_path)

diagnostic_rows = []
for name, path in required_input_paths.items():
    path = Path(path)
    exists = path.exists()
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': f'input_exists__{name}',
        'path': str(path),
        'required': True,
        'check_passed': exists,
        'file_exists': exists,
        'warning_message': '' if exists else 'Required input missing.',
    })
for name, path in optional_input_paths.items():
    path = Path(path)
    exists = path.exists()
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': f'input_exists__{name}',
        'path': str(path),
        'required': False,
        'check_passed': True,
        'file_exists': exists,
        'warning_message': '' if exists else 'Optional input not found; not required for this run.',
    })
for path in saved_paths:
    path = Path(path)
    exists = path.exists()
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': f'output_written__{path.name}',
        'path': str(path),
        'required': True,
        'check_passed': exists,
        'file_exists': exists,
        'warning_message': '' if exists else 'Expected output missing.',
    })

diagnostic_rows.append({
    'stage': STAGE,
    'check_name': 'no_prior_model_feature_contract_clean',
    'path': '',
    'required': True,
    'check_passed': bool(feature_leakage_qc_df.loc[feature_leakage_qc_df['check'].eq('no_prior_model_feature_contract_clean'), 'passed'].all()) if 'feature_leakage_qc_df' in globals() else False,
    'file_exists': True,
    'warning_message': '',
})
diagnostic_rows.append({
    'stage': STAGE,
    'check_name': 'no_prior_score_components_disabled',
    'path': '',
    'required': True,
    'check_passed': bool(feature_leakage_qc_df.loc[feature_leakage_qc_df['check'].eq('no_prior_score_components_disabled'), 'passed'].all()) if 'feature_leakage_qc_df' in globals() else False,
    'file_exists': True,
    'warning_message': '',
})
diagnostic_rows.append({
    'stage': STAGE,
    'check_name': 'no_prior_shannon_ranking_matches_stage1_baseline',
    'path': '',
    'required': True,
    'check_passed': bool(no_prior_shannon_ranking_matches_stage1_baseline),
    'file_exists': True,
    'warning_message': '' if no_prior_shannon_ranking_matches_stage1_baseline else 'No-prior Shannon ranking differs from stage1_baseline.',
})

diagnostic_rows.append({
    'stage': STAGE,
    'check_name': 'candidate_set_unchanged_by_reranker',
    'path': '',
    'required': True,
    'check_passed': not candidate_set_changed_by_reranker,
    'file_exists': True,
    'warning_message': '' if not candidate_set_changed_by_reranker else 'Candidate set changed after reranking.',
})
metadata_qc_checks = {
    'retrieval_method_matches_expected': retrieval_method_actual == DEFAULT_RETRIEVAL_METHOD,
    'retrieval_method_label_matches_expected': retrieval_method_label_actual == DEFAULT_RETRIEVAL_METHOD_LABEL,
    'candidate_pool_type_matches_expected': candidate_pool_type_actual == DEFAULT_CANDIDATE_POOL_TYPE,
    'reranking_method_is_shannon_no_prior': RERANKING_METHOD == "shannon_no_prior",
    'rerank_method_is_shannon_no_prior_rerank': SHANNON_OUTPUT_RERANK_METHOD == "shannon_no_prior_rerank",
}
for check_name, check_passed in metadata_qc_checks.items():
    diagnostic_rows.append({
        'stage': STAGE,
        'check_name': check_name,
        'path': '',
        'required': True,
        'check_passed': bool(check_passed),
        'file_exists': True,
        'warning_message': '' if check_passed else 'Metadata value did not match expected constant.',
    })
diagnostics_summary_df = pd.DataFrame(diagnostic_rows)
diagnostics_summary_df.to_csv(diagnostics_summary_path, index=False)
diagnostics_exists = diagnostics_summary_path.exists()
diagnostic_rows.append({
    'stage': STAGE,
    'check_name': f'output_written__{diagnostics_summary_path.name}',
    'path': str(diagnostics_summary_path),
    'required': True,
    'check_passed': diagnostics_exists,
    'file_exists': diagnostics_exists,
    'warning_message': '' if diagnostics_exists else 'Expected output missing.',
})
diagnostics_summary_df = pd.DataFrame(diagnostic_rows)
diagnostics_summary_df.to_csv(diagnostics_summary_path, index=False)
saved_paths.append(diagnostics_summary_path)

stale_terms = [
    'tuned_' + 'graph_hybrid',
    'baseline_' + 'tuned_' + 'graph_hybrid',
    'Tuned ' + 'Graph-Hybrid',
]

search_payload = json.dumps(make_jsonable({
    'config_snapshot': config_snapshot if 'config_snapshot' in globals() else {},
    'run_manifest': run_manifest if 'run_manifest' in globals() else {},
}), ensure_ascii=False)

stale_found = [term for term in stale_terms if term in search_payload]
if stale_found:
    raise RuntimeError(f"Stale tuned graph-hybrid metadata found in Notebook 10 outputs: {stale_found}")

print('Saved outputs:')
for path in saved_paths:
    print('-', path)


# Refresh the exported contracts with Notebook 09 winner lineage.
notebook09_lineage = {
    "notebook_09_dependency": True,
    "stage1_candidate_pool_manifest_path": str(CANDIDATE_POOL_MANIFEST_PATH),
    "candidate_pool_role": CANDIDATE_POOL_ROLE,
    "candidate_retrieval_method_key": CANDIDATE_RETRIEVAL_METHOD_KEY,
    "candidate_retrieval_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "source_baseline_method_key": CANDIDATE_RETRIEVAL_METHOD_KEY,
    "source_baseline_method_label": CANDIDATE_RETRIEVAL_METHOD_LABEL,
    "candidate_budget_policy": stage1_pool_manifest.get("candidate_budget_policy"),
    "candidate_budget_k": int(stage1_pool_manifest.get("candidate_budget_k", POOL_K)),
    "exact_k_validation_passed": not bool(stage1_pool_manifest.get("variable_candidate_count_allowed", True)),
    "user_prior_feature_policy": USER_PRIOR_FEATURE_POLICY,
    "user_prior_features_enabled": bool(USE_USER_PRIOR_FEATURES),
    "strict_prior_history_loaded": bool(USE_USER_PRIOR_FEATURES),
    "prior_history_loaded": bool(USE_USER_PRIOR_FEATURES),
    "candidate_source_qc": candidate_source_qc,
}
config_snapshot.update(notebook09_lineage)
Path(config_snapshot_path).write_text(
    json.dumps(make_jsonable(config_snapshot), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
run_manifest.update(notebook09_lineage)
run_manifest.setdefault("input_paths", {})["stage1_candidate_pool_manifest"] = str(CANDIDATE_POOL_MANIFEST_PATH)
run_manifest["input_paths"]["winner_candidate_pool"] = str(CANDIDATE_POOL_PATH)
run_manifest["input_paths"]["query_cache_parquet"] = str(QUERY_CACHE_PARQUET)
if not USE_USER_PRIOR_FEATURES:
    run_manifest["input_paths"]["prior_history_parquet"] = None
Path(run_manifest_path).write_text(
    json.dumps(make_jsonable(run_manifest), ensure_ascii=False, indent=2),
    encoding="utf-8",
)


Saved outputs:
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/results_overall.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/results_by_pool_depth.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/results_by_regime.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/results_by_regime_pool_depth.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/runtime_by_pool_depth.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/uplift_summary.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonal

17600

In [21]:

# =========================================================
# Feature Contract and Temporal Diagnostics Export
# =========================================================
feature_manifest_path = OUT_DIR / "feature_manifest.json"
used_model_features_path = OUT_DIR / "used_model_features.csv"
feature_leakage_qc_path = OUT_DIR / "feature_leakage_qc.csv"
temporal_feature_summary_path = OUT_DIR / "temporal_feature_summary.csv"
sample_count_qc_path = OUT_DIR / "sample_count_qc.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if "feature_manifest" not in globals():
    feature_manifest = {
        "notebook_name": NOTEBOOK_NAME,
        "stage": STAGE,
        "reranking_method": RERANKING_METHOD,
        "temporal_feature_version": globals().get("TEMPORAL_FEATURE_VERSION", "not_available"),
        "raw_timestamp_features_used": False,
    }

if "query_brand_leak_qc_source" in globals():
    feature_manifest["query_brand_leak_qc_source"] = query_brand_leak_qc_source
    feature_manifest["query_brand_leak_row_count"] = int(brand_terms_added_to_synthetic_query_n)

with open(feature_manifest_path, "w", encoding="utf-8") as f:
    json.dump(make_jsonable(feature_manifest) if "make_jsonable" in globals() else feature_manifest, f, ensure_ascii=False, indent=2)

if "used_model_features_df" in globals():
    used_model_features_df.to_csv(used_model_features_path, index=False)
else:
    pd.DataFrame([{"model": RERANKING_METHOD, "feature": "not_recorded"}]).to_csv(used_model_features_path, index=False)

if "feature_leakage_qc_df" in globals():
    feature_leakage_qc_df.to_csv(feature_leakage_qc_path, index=False)
else:
    pd.DataFrame([{"check": "not_recorded", "passed": False}]).to_csv(feature_leakage_qc_path, index=False)

if "temporal_feature_summary_df" in globals():
    temporal_feature_summary_df.to_csv(temporal_feature_summary_path, index=False)
else:
    pd.DataFrame([{"feature": "not_available"}]).to_csv(temporal_feature_summary_path, index=False)

if "sample_count_qc_df" in globals():
    sample_count_qc_df.to_csv(sample_count_qc_path, index=False)

print("Feature contract exports written:")
print("-", feature_manifest_path)
print("-", used_model_features_path)
print("-", feature_leakage_qc_path)
print("-", temporal_feature_summary_path)
print("-", sample_count_qc_path)


# Persist the executed brand/parity contract in the existing manifests.
if Path(config_snapshot_path).exists():
    config_contract = load_json_if_exists(config_snapshot_path)
    config_contract.update({
        **brand_contract,
        "scoring_components": list(SHANNON_SCORING_COMPONENTS),
        "feature_columns": list(SHANNON_FEATURE_COLUMNS),
        "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
        "contract_diagnostics": contract_diagnostics,
    })
    Path(config_snapshot_path).write_text(json.dumps(make_jsonable(config_contract), ensure_ascii=False, indent=2), encoding="utf-8")

if Path(run_manifest_path).exists():
    run_contract = load_json_if_exists(run_manifest_path)
    run_contract.update({
        **brand_contract,
        "scoring_components": list(SHANNON_SCORING_COMPONENTS),
        "feature_columns": list(SHANNON_FEATURE_COLUMNS),
        "feature_dtypes": dict(SHANNON_FEATURE_DTYPES),
        "contract_diagnostics": contract_diagnostics,
        "shared_all_prior_contract_path": str(SHARED_ALL_PRIOR_CONTRACT_PATH) if USE_USER_PRIOR_FEATURES else None,
    })
    Path(run_manifest_path).write_text(json.dumps(make_jsonable(run_contract), ensure_ascii=False, indent=2), encoding="utf-8")


Feature contract exports written:
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/feature_manifest.json
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/used_model_features.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/feature_leakage_qc.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/temporal_feature_summary.csv
- /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/sample_count_qc.csv


## 10. Final Contract Validation

In [22]:
expected_outputs = [
    results_overall_path,
    results_by_pool_depth_path,
    results_by_regime_path,
    results_by_regime_pool_depth_path,
    runtime_by_pool_depth_path,
    uplift_summary_path,
    feature_table_path,
    per_query_results_path,
    reranked_candidates_path,
    query_profiles_path,
    OUT_DIR / 'runtime_steps.csv',
    OUT_DIR / 'runtime_notebook_summary.csv',
    OUT_DIR / 'runtime_method_summary_at1000.csv',
    OUT_DIR / 'runtime_pool_depth_diagnostic.csv',
    OUT_DIR / 'runtime_method_components_at1000_herbal.csv',
    config_snapshot_path,
    diagnostics_summary_path,
    run_manifest_path,
]
missing_outputs = [str(path) for path in expected_outputs if not Path(path).exists()]
if missing_outputs:
    raise RuntimeError(f'Missing expected outputs: {missing_outputs}')

if sorted(summary_overall_df['rerank_method'].astype(str).unique().tolist()) != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError('results_overall methods do not match EXPECTED_RERANK_METHODS.')
if sorted(per_query_results_df['rerank_method'].astype(str).unique().tolist()) != sorted(EXPECTED_RERANK_METHODS):
    raise RuntimeError('per_query_metrics methods do not match EXPECTED_RERANK_METHODS.')
if sorted(pd.to_numeric(per_query_results_df['pool_depth'], errors='coerce').dropna().astype(int).unique().tolist()) != sorted(POOL_DEPTHS):
    raise RuntimeError('per_query_metrics does not preserve all configured POOL_DEPTHS.')
if candidate_set_changed_by_reranker:
    raise RuntimeError('Candidate set was changed by Shannon reranker.')
if not no_prior_shannon_ranking_matches_stage1_baseline:
    raise RuntimeError('No-prior Shannon rank order must match stage1_baseline within each query and pool_depth.')

print('Final validation passed.')
print('Primary Stage 2 metric:', PRIMARY_STAGE2_METRIC)
display(summary_overall_df[['rerank_method', 'n_queries', 'NDCG@5', 'HitRate@5', 'MRR@5'] + [c for c in ['weighted_facet_overlap_at_5'] if c in summary_overall_df.columns]])
display(runtime_method_summary_at1000_df)

final_qc = {
    "candidate_input_path": str(CANDIDATE_POOL_PATH),
    "query_count": int(query_meta_df["query_id"].nunique()),
    "candidate_row_count": int(len(candidate_df)),
    "regime_counts": query_meta_df["regime"].astype(str).value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict(),
    "max_target_rank_desc": int(pd.to_numeric(query_meta_df["target_rank_desc"], errors="coerce").max()) if "target_rank_desc" in query_meta_df.columns and query_meta_df["target_rank_desc"].notna().any() else None,
    "unique_target_selection_mode": sorted(query_meta_df["target_selection_mode"].dropna().astype(str).unique().tolist()) if "target_selection_mode" in query_meta_df.columns else [],
    "missing_target_timestamp_ms_count": int(query_meta_df["target_timestamp_ms"].isna().sum()) if "target_timestamp_ms" in query_meta_df.columns else int(len(query_meta_df)),
    "output_folder": str(OUT_DIR),
}
print("Candidate input path:", final_qc["candidate_input_path"])
print("Query count:", final_qc["query_count"])
print("Candidate row count:", final_qc["candidate_row_count"])
print("Regime counts:", final_qc["regime_counts"])
print("Max target_rank_desc:", final_qc["max_target_rank_desc"])
print("Unique target_selection_mode:", final_qc["unique_target_selection_mode"])
print("Missing target_timestamp_ms count:", final_qc["missing_target_timestamp_ms_count"])
print("Output folder:", final_qc["output_folder"])

# Final benchmark compatibility checks
expected_query_count_final = (
    int(EXPECTED_QUERY_COUNT)
    if EXPECTED_QUERY_COUNT is not None
    else int(query_meta_df["query_id"].nunique())
)
expected_candidate_rows_final = (
    int(EXPECTED_CANDIDATE_ROWS_TOP1000)
    if EXPECTED_CANDIDATE_ROWS_TOP1000 is not None
    else int(len(candidate_df))
)

if final_qc["query_count"] != expected_query_count_final:
    raise RuntimeError(f"Query count mismatch: {final_qc['query_count']} vs {expected_query_count_final}")
if final_qc["candidate_row_count"] != expected_candidate_rows_final:
    raise RuntimeError(f"Candidate row count mismatch: {final_qc['candidate_row_count']} vs{expected_candidate_rows_final}")

Final validation passed.
Primary Stage 2 metric: NDCG@5


,rerank_method,n_queries,NDCG@5,HitRate@5,MRR@5,weighted_facet_overlap_at_5
0,shannon_no_prior_rerank,1968,0.023847,0.037093,0.019521,0.024726
1,stage1_baseline,1968,0.023847,0.037093,0.019521,0.024726


,category,category_id,notebook_name,branch,method,rerank_method,pool_depth,n_queries,n_candidates,online_operation_runtime_sec,offline_preparation_runtime_sec,evaluation_export_runtime_sec,total_notebook_runtime_sec,runtime_sec_per_query,runtime_sec_per_candidate,queries_per_second,candidates_per_second,runtime_scope,candidate_pool_type,retrieval_method,retrieval_method_label,reranking_method,sample_scope,common_sample_filtering_introduced,primary_metric,ndcg_at_5,hitrate_at_5,mrr_at_5,baseline_ndcg_at_5,baseline_hitrate_at_5,delta_ndcg_at_5_vs_baseline,delta_hitrate_at_5_vs_baseline,delta_ndcg5_per_100sec_online,delta_hitrate5_per_100sec_online
0,herbal_supplements,herbal,10a_no_prior_rerank_shannon_entropy_herbal.ipynb,baseline_winner_shannon_no_prior,shannon_no_prior,shannon_no_prior_rerank,1000,1968.0,1968000.0,11.502817,0.0,19.117034,146.393042,0.005845,0.000006,171.08852,171088.519687,stage2_online_reranking_excluding_stage1_retri...,Dense-BM25 Hybrid,Dense-BM25 Hybrid,Dense-BM25 Hybrid,shannon_no_prior,native,False,NDCG@5,0.023847,0.037093,0.019521,0.023847,0.037093,0.0,0.0,0.0,0.0


Candidate input path: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/by_method/query_only_winner_top1000_herbal.parquet
Query count: 1968
Candidate row count: 1968000
Regime counts: {'cold': 656, 'weak': 656, 'strong': 656}
Max target_rank_desc: 5
Unique target_selection_mode: ['recent_eligible_review_rank_le5']
Missing target_timestamp_ms count: 0
Output folder: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior


In [23]:
# =========================================================
# Notebook 09 Winner-Lineage QC
# =========================================================
expected_path_key = (
    "query_only_winner_long"
    if CANDIDATE_POOL_ROLE == "baseline_query_only"
    else "personalized_winner_long"
)
expected_candidate_path = Path(stage1_pool_manifest["output_paths"][expected_path_key])
expected_method_slug = (
    stage1_pool_manifest["baseline_retrieval_winner_method_key"]
    if CANDIDATE_POOL_ROLE == "baseline_query_only"
    else stage1_pool_manifest["selected_personalized_method_slug"]
)
expected_method_label = (
    stage1_pool_manifest["baseline_retrieval_winner_method_label"]
    if CANDIDATE_POOL_ROLE == "baseline_query_only"
    else stage1_pool_manifest["selected_personalized_method_label"]
)

winner_lineage_qc_df = pd.DataFrame([
    {
        "check": "candidate_path_matches_notebook09",
        "passed": Path(CANDIDATE_POOL_PATH) == expected_candidate_path,
        "observed": str(CANDIDATE_POOL_PATH),
        "expected": str(expected_candidate_path),
    },
    {
        "check": "candidate_role_matches_condition",
        "passed": candidate_df["candidate_pool_role"].astype(str).eq(CANDIDATE_POOL_ROLE).all(),
        "observed": ",".join(sorted(candidate_df["candidate_pool_role"].astype(str).unique().tolist())),
        "expected": CANDIDATE_POOL_ROLE,
    },
    {
        "check": "winner_method_slug_matches_notebook09",
        "passed": CANDIDATE_RETRIEVAL_METHOD_KEY == expected_method_slug,
        "observed": CANDIDATE_RETRIEVAL_METHOD_KEY,
        "expected": expected_method_slug,
    },
    {
        "check": "winner_method_label_matches_notebook09",
        "passed": CANDIDATE_RETRIEVAL_METHOD_LABEL == expected_method_label,
        "observed": CANDIDATE_RETRIEVAL_METHOD_LABEL,
        "expected": expected_method_label,
    },
    {
        "check": "exact_k_candidate_count",
        "passed": candidate_df.groupby("query_id")["item_id"].size().eq(POOL_K).all(),
        "observed": f"{candidate_df.groupby('query_id')['item_id'].size().min()}..{candidate_df.groupby('query_id')['item_id'].size().max()}",
        "expected": str(POOL_K),
    },
    {
        "check": "user_prior_policy_matches_condition",
        "passed": bool(USE_USER_PRIOR_FEATURES) == (EXPERIMENT_CONDITION != "s2q_no_prior_reranking"),
        "observed": str(bool(USE_USER_PRIOR_FEATURES)),
        "expected": str(EXPERIMENT_CONDITION != "s2q_no_prior_reranking"),
    },
])

winner_lineage_qc_path = OUT_DIR / f"winner_candidate_source_qc_{CATEGORY_ID}.csv"
winner_lineage_qc_df.to_csv(winner_lineage_qc_path, index=False)
print("Winner candidate source QC:")
display(winner_lineage_qc_df)
print("Output:", winner_lineage_qc_path)

if not winner_lineage_qc_df["passed"].all():
    failed = winner_lineage_qc_df.loc[~winner_lineage_qc_df["passed"], "check"].tolist()
    raise RuntimeError(f"Notebook 09 winner-lineage QC failed: {failed}")


Winner candidate source QC:


,check,passed,observed,expected
0,candidate_path_matches_notebook09,True,/content/drive/MyDrive/thesis_recsys/categorie...,/content/drive/MyDrive/thesis_recsys/categorie...
1,candidate_role_matches_condition,True,baseline_query_only,baseline_query_only
2,winner_method_slug_matches_notebook09,True,hybrid_dense_bm25,hybrid_dense_bm25
3,winner_method_label_matches_notebook09,True,Dense-BM25 Hybrid,Dense-BM25 Hybrid
4,exact_k_candidate_count,True,1000..1000,1000
5,user_prior_policy_matches_condition,True,False,False


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/winner_candidate_source_qc_herbal.csv


In [24]:
# =========================================================
# Runtime Pool Depth QC
# =========================================================
expected_runtime_depths = sorted(int(depth) for depth in POOL_DEPTHS)
runtime_depths_present = sorted(runtime_by_pool_depth_df["pool_depth"].astype(int).unique().tolist())
runtime_component_columns = [
    "feature_preparation_runtime_sec",
    "model_scoring_runtime_sec",
    "ranking_sorting_runtime_sec",
    "online_operation_runtime_sec",
    "runtime_sec_per_query",
    "runtime_sec_per_candidate",
    "queries_per_second",
    "candidates_per_second",
]
missing_runtime_components = [
    column for column in runtime_component_columns
    if column not in runtime_by_pool_depth_df.columns
]
duplicate_key_columns = [
    column for column in ["pool_depth", "reranker", "method", "reranking_method"]
    if column in runtime_by_pool_depth_df.columns
]
duplicate_runtime_rows = int(runtime_by_pool_depth_df.duplicated(duplicate_key_columns).sum()) if duplicate_key_columns else 0
runtime_file_candidates = []

for _name in [
    "runtime_by_pool_depth_path",
    "runtime_report_summary_path",
    "runtime_steps_path",
    "runtime_notebook_summary_path",
]:
    if _name in globals():
        runtime_file_candidates.append(globals()[_name])

if "output_paths" in globals() and isinstance(output_paths, dict):
    runtime_file_candidates.extend(
        path for key, path in output_paths.items()
        if str(key).startswith("runtime") or str(key).endswith("manifest")
    )

runtime_files_created = sorted({
    str(Path(path)) for path in runtime_file_candidates
    if path is not None and Path(path).exists()
})


print("Candidate pool depths evaluated:", expected_runtime_depths)
print("Candidate pool depths in runtime_by_pool_depth.csv:", runtime_depths_present)
print("Every evaluated pool depth has a runtime row:", runtime_depths_present == expected_runtime_depths)
print("Output runtime files created:", runtime_files_created)
if missing_runtime_components:
    print("Warning: missing runtime component columns:", missing_runtime_components)
else:
    print("Warning: missing runtime component columns: none")
if duplicate_runtime_rows:
    print("Warning: duplicate runtime rows:", duplicate_runtime_rows)
else:
    print("Warning: duplicate runtime rows: none")

if runtime_depths_present != expected_runtime_depths:
    raise RuntimeError("Not every evaluated pool depth has a runtime row.")
if missing_runtime_components:
    raise RuntimeError("Runtime component columns are missing from runtime_by_pool_depth_df.")
if duplicate_runtime_rows:
    raise RuntimeError("Duplicate per-depth runtime rows were found.")


Candidate pool depths evaluated: [100, 300, 500, 700, 1000]
Candidate pool depths in runtime_by_pool_depth.csv: [100, 300, 500, 700, 1000]
Every evaluated pool depth has a runtime row: True
Output runtime files created: ['/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/runtime_by_pool_depth.csv']


In [25]:
# =========================================================
# Metric Depth QC
# =========================================================
_required_eval_ks = [1, 5, 10, 100, 300, 500, 700, 1000]
_required_pool_depths = [100, 300, 500, 700, 1000]
_metric_qc_df = globals().get("per_query_results_df", globals().get("per_query_metrics_df", pd.DataFrame()))
_summary_qc_df = globals().get("summary_by_pool_depth_df", globals().get("results_by_pool_depth_df", pd.DataFrame()))
_available_hitrate_cols = sorted([col for col in _metric_qc_df.columns if re.match(r"^HitRate@\d+$", str(col))], key=lambda col: int(str(col).split("@")[1]))
_available_ndcg_cols = sorted([col for col in _metric_qc_df.columns if re.match(r"^NDCG@\d+$", str(col))], key=lambda col: int(str(col).split("@")[1]))
_available_mrr_cols = sorted([col for col in _metric_qc_df.columns if re.match(r"^MRR@\d+$", str(col))], key=lambda col: int(str(col).split("@")[1]))
_required_metric_cols = [
    f"{metric}@{k}"
    for metric in ["HitRate", "NDCG", "MRR"]
    for k in _required_eval_ks
]
_missing_metric_cols = [col for col in _required_metric_cols if col not in _metric_qc_df.columns]
_pool_depths_present = sorted(pd.to_numeric(_metric_qc_df.get("pool_depth", pd.Series(dtype="float64")), errors="coerce").dropna().astype(int).unique().tolist())
_summary_missing_metric_cols = [col for col in _required_metric_cols if col not in _summary_qc_df.columns]
_every_pool_depth_has_metrics = (
    _pool_depths_present == _required_pool_depths
    and not _missing_metric_cols
    and not _summary_missing_metric_cols
)
_runtime_depths_present = []
if isinstance(globals().get("runtime_by_pool_depth_df"), pd.DataFrame) and "pool_depth" in runtime_by_pool_depth_df.columns:
    _runtime_depths_present = sorted(pd.to_numeric(runtime_by_pool_depth_df["pool_depth"], errors="coerce").dropna().astype(int).unique().tolist())
_output_files_created = []
for _name, _value in list(globals().items()):
    if _name.endswith("_path") and isinstance(_value, (str, Path)):
        _path = Path(_value)
        if _path.exists():
            _output_files_created.append(str(_path))
if "output_paths" in globals() and isinstance(output_paths, dict):
    for _path in output_paths.values():
        _path = Path(_path)
        if _path.exists():
            _output_files_created.append(str(_path))
_output_files_created = sorted(set(_output_files_created))

print("POOL_DEPTHS:", list(POOL_DEPTHS))
print("EVAL_KS:", list(EVAL_KS))
print("Available HitRate columns:", _available_hitrate_cols)
print("Available NDCG columns:", _available_ndcg_cols)
print("Available MRR columns:", _available_mrr_cols)
print("Evaluated pool depths:", _pool_depths_present)
print("Runtime pool depths:", _runtime_depths_present)
print("Every evaluated pool depth has all required EVAL_KS metrics:", _every_pool_depth_has_metrics)
print("Output files created:", _output_files_created)

if list(POOL_DEPTHS) != _required_pool_depths:
    raise RuntimeError(f"POOL_DEPTHS mismatch: {list(POOL_DEPTHS)}")
if list(EVAL_KS) != _required_eval_ks:
    raise RuntimeError(f"EVAL_KS mismatch: {list(EVAL_KS)}")
if _pool_depths_present != _required_pool_depths:
    raise RuntimeError(f"Metric rows do not cover required pool depths: {_pool_depths_present}")
if _missing_metric_cols:
    raise RuntimeError(f"Missing per-query metric columns: {_missing_metric_cols}")
if _summary_missing_metric_cols:
    raise RuntimeError(f"Missing by-pool-depth summary metric columns: {_summary_missing_metric_cols}")


POOL_DEPTHS: [100, 300, 500, 700, 1000]
EVAL_KS: [1, 5, 10, 100, 300, 500, 700, 1000]
Available HitRate columns: ['HitRate@1', 'HitRate@5', 'HitRate@10', 'HitRate@100', 'HitRate@300', 'HitRate@500', 'HitRate@700', 'HitRate@1000']
Available NDCG columns: ['NDCG@1', 'NDCG@5', 'NDCG@10', 'NDCG@100', 'NDCG@300', 'NDCG@500', 'NDCG@700', 'NDCG@1000']
Available MRR columns: ['MRR@1', 'MRR@5', 'MRR@10', 'MRR@100', 'MRR@300', 'MRR@500', 'MRR@700', 'MRR@1000']
Evaluated pool depths: [100, 300, 500, 700, 1000]
Runtime pool depths: [100, 300, 500, 700, 1000]
Every evaluated pool depth has all required EVAL_KS metrics: True
Output files created: ['/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/by_method/query_only_winner_top1000_herbal.parquet', '/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/shannon_no_prior/config_snapshot.json', '/content/drive/MyDrive/thesis_recsys/categories/herbal_supp